# Intro

Welcome to the interactive tracking notebook!\
This notebook goes through each step and allows you to tune parameters and view how it changes the results.

The notebook proceeds as follows:
1. **Import** libraries
2. Define **paths** to data
3. Run data through the **pipeline**. Each step of the pipeline is run by a single unique python class.
4. **Visualize** results
5. **Save** results

As you go through the notebook, take note of the small number of parameters that are mentioned as **'important parameters'** (consider searching for these in the notebook). We consider these to be the only parameters that can have a large effect on the run output. Other parameters matter and should be considered as well, but are less critical.

##### If running on google colab:

- install roicat

After running the cell below, the runtime will restart.

In [2]:
#@title Install `roicat` if on colab
using_colab = 'google.colab' in str(get_ipython())

if using_colab:
    !pip uninstall -y tensorflow
    !pip install roicat[tracking]

- mount google drive

In [3]:
#@title mount gdrive if on colab
#@markdown Upload your data to Google Drive, then mount the drive and access the cloud directory here.
#@markdown You can use the sidebar to the left to browse your google drive directories.

using_colab = 'google.colab' in str(get_ipython())

if using_colab:
    from google.colab import drive
    path_gdrive = '/content/gdrive'
    drive.mount(path_gdrive, force_remount=True)

- enable widgets

In [4]:
if using_colab:
    from google.colab import output
    output.enable_custom_widget_manager()

# Import libraries

widen the notebook

In [5]:
# widen jupyter notebook window
from IPython.display import display, HTML
display(HTML("<style>.container {width:95% !important; }</style>"))
display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

Import basic libraries

In [6]:
from pathlib import Path
import multiprocessing as mp
import tempfile

import numpy as np

Import `roicat`

In [7]:
# %load_ext autoreload
# %autoreload 2
import roicat

# Find paths to data

In this example we are using suite2p output files, but other data types can be used (CaImAn, etc.) \
See the notebook on ingesting diverse data: https://github.com/RichieHakim/ROICaT/blob/main/notebooks/jupyter/other/demo_data_importing.ipynb

Make a list containing the paths to all the input files.

In this example we are using suite2p, so the following are defined:
1. `paths_allStat`: a list to all the stat.npy files
2. `paths_allOps`: a list with ops.npy files that correspond 1-to-1 with the stat.npy files

In [8]:
dir_allOuterFolders = r'/Users/richardhakim/Documents/data/mouse_0322R_stat_and_ops/stat_and_ops'

pathSuffixToStat = 'stat.npy'
pathSuffixToOps = 'ops.npy'

paths_allStat = roicat.helpers.find_paths(
    dir_outer=dir_allOuterFolders,
    reMatch=pathSuffixToStat,
    depth=10,
)[:]
paths_allOps  = [str(Path(path).resolve().parent / pathSuffixToOps) for path in paths_allStat]

print(f'paths to all stat files:');
[print(path) for path in paths_allStat];
print('');
print(f'paths to all ops files:');
[print(path) for path in paths_allOps];

paths to all stat files:
/Users/richardhakim/Documents/data/mouse_0322R_stat_and_ops/stat_and_ops/20230424/stat.npy
/Users/richardhakim/Documents/data/mouse_0322R_stat_and_ops/stat_and_ops/20230425/stat.npy
/Users/richardhakim/Documents/data/mouse_0322R_stat_and_ops/stat_and_ops/20230426/stat.npy
/Users/richardhakim/Documents/data/mouse_0322R_stat_and_ops/stat_and_ops/20230427/stat.npy
/Users/richardhakim/Documents/data/mouse_0322R_stat_and_ops/stat_and_ops/20230428/stat.npy
/Users/richardhakim/Documents/data/mouse_0322R_stat_and_ops/stat_and_ops/20230429/stat.npy
/Users/richardhakim/Documents/data/mouse_0322R_stat_and_ops/stat_and_ops/20230430/stat.npy
/Users/richardhakim/Documents/data/mouse_0322R_stat_and_ops/stat_and_ops/20230501/stat.npy
/Users/richardhakim/Documents/data/mouse_0322R_stat_and_ops/stat_and_ops/20230502/stat.npy
/Users/richardhakim/Documents/data/mouse_0322R_stat_and_ops/stat_and_ops/20230503/stat.npy
/Users/richardhakim/Documents/data/mouse_0322R_stat_and_ops/stat_

# Import data

**Important parameters**:
- `um_per_pixel` (float):
    - Resolution. 'micrometers per pixel' of the imaging field of view.

In [12]:
data = roicat.data_importing.Data_suite2p(
    paths_statFiles=paths_allStat[:],
    paths_opsFiles=paths_allOps[:],
    um_per_pixel=1.5,  ## IMPORTANT PARAMETER. Use a list of floats if values differ in each session.
    new_or_old_suite2p='new',
    type_meanImg='meanImgE',
    verbose=True,
)

assert data.check_completeness(verbose=False)['tracking'], f"Data object is missing attributes necessary for tracking."

Starting: Importing FOV images from ops files
Completed: Set FOV_height and FOV_width successfully.
Completed: Imported 35 FOV images.
Setting FOV_images...
Completed: Set FOV_images for 35 sessions successfully.
Importing spatial footprints from stat files.


  0%|          | 0/35 [00:00<?, ?it/s]

Imported 35 sessions of spatial footprints into sparse arrays.
Completed: Created session_bool.
Completed: Set spatialFootprints for 35 sessions successfully.
Completed: Created centroids.
Starting: Creating centered ROI images from spatial footprints...
Completed: Created ROI images.


**Note on subselecting ROIs**:

Generally, I recommend the following:
- Do not subselecting ROIs prior to tracking. Use all available ROIs for the entire tracking process.
- Then, after tracking, subsequently apply classification and inclusion criteria to remove bad ROIs. See the 'tracking_handling_outputs.ipynb' notebook for details.

However, if you want to subselect ROIs before tracking you can call `data.remove_ROIs_by_classLabel()`. Prior to doing this, you will first need to set the class labels in the `data` object by either calling `data.set_class_labels()` or providing the `paths_iscell` input argument (if using the `Data_suite2p` class). Code below:

```python
data.remove_rois_by_classLabel(classLabel_to_keep=1, verbose=True)
```

In [ ]:
roicat.visualization.display_toggle_image_stack(data.FOV_images)
roicat.visualization.display_toggle_image_stack(data.get_maxIntensityProjection_spatialFootprints(), clim=[0,1])
roicat.visualization.display_toggle_image_stack(np.concatenate(data.ROI_images, axis=0)[:5000], image_size=(200,200))

### Set the device to run on

If you have a GPU, some steps can be sped up.

In [ ]:
DEVICE = roicat.helpers.set_device(use_GPU=True)

### Set determinism and seed
Perfect determinism is not possible for a variety of reasons, but by setting `deterministic` to True you can get close by forcing the backend code to use deterministic algorithms where possible. It is generally not recommended to use deterministic algorithms and seeds due to the potential for reproducing low probability outputs and slower computation. Though, it can be useful in recreating bugs. If you want to do so, you can set the `deterministic` parameter to `True` below.

In [ ]:
SEED = roicat.util.set_random_seed(seed=None, deterministic=False)  ## Deterministic algorithms have issues, but are useful for debugging, testing, and reproducing results.

# Alignment

This is the most important step in the pipeline to stop and check that everything looks okay and tune parameters if necessary.

Alignment is 4 steps:

1. FOV_image augmentation
2. Fit geometric transformation
3. Fit non-rigid transformation (on top of the geometric)
4. Apply transformation to ROIs

##### 1. FOV_image augmentation
Do what is necessary to make the augmented FOV_images look good. Use the visualization tool below to help. This can include playing with the mixing factor, normalization, and playing with the CLAHE parameters.

In [ ]:
aligner = roicat.tracking.alignment.Aligner(
    use_match_search=True,  ## Use our algorithm for doing all-pairs matching if template matching fails.
    all_to_all=False,  ## Force the use of our algorithm for all-pairs matching. Much slower (False: O(N) vs. True: O(N^2)), but more accurate.
    radius_in=4,  ## IMPORTANT PARAMETER: Value in micrometers used to define the maximum shift/offset between two images that are considered to be aligned. Larger means more lenient alignment.
    radius_out=20,  ## Value in micrometers used to define the minimum shift/offset between two images that are considered to be misaligned.
    z_threshold=4.0,  ## IMPORTANT PARAMETER: Z-score required to define two images as aligned. Larger values results in more stringent alignment requirements.
    um_per_pixel=data.um_per_pixel[0],  ## Single value for um_per_pixel. data.um_per_pixel is typically a list of floats, so index out just one value.
    device=DEVICE,
    verbose=True,
)

In [ ]:
FOV_images = aligner.augment_FOV_images(
    FOV_images=data.FOV_images,
    spatialFootprints=data.spatialFootprints,
    normalize_FOV_intensities=True,
    roi_FOV_mixing_factor=0.5,
    use_CLAHE=True,  ## IMPORTANT PARAMETER. Use Set to False if data is poor quality or poorly aligned.
    CLAHE_grid_block_size=10,  ## IMPORTANT PARAMETER. Use smaller values for higher precision but higher chance of failure.
    CLAHE_clipLimit=1.0,
    CLAHE_normalize=True,
)

View the augmented FOV images

In [ ]:
roicat.visualization.display_toggle_image_stack(FOV_images)

##### 2. Fit geometric transformation
This is an important step. Consider reading the comments and arguments closely.

Play with parameters until the aligned FOV_images look good. The visualization tool below can help.

This step creates the attribute: `aligner.ims_registered_geo`, which are the registered images after the geometric transformation.

We like the following **important parameters**:
- `template=0.5`
- `method='RoMa'` if you have a GPU, otherwise use one of the other methods
- `template_method='sequential'` best if there are large changes that occurrs slowly over sessions

In [ ]:
aligner.fit_geometric(
    template=0.5,  ## specifies which image to use as the template. Either array (image), integer (ims_moving index), or float (ims_moving fractional index)
    ims_moving=FOV_images,  ## input images
    template_method='sequential',  ## 'sequential': align images to neighboring images (good for drifting data). 'image': align to a single image
    mask_borders=(0, 0, 0, 0),  ## number of pixels to mask off the edges (top, bottom, left, right)
    method='DISK_LightGlue',  ## See below for options.
    kwargs_method = {
        'RoMa': {  ## Accuracy: Best, Speed: Very slow (can be fast with a GPU).
            'model_type': 'outdoor',
            'n_points': 10000,  ## Higher values mean more points are used for the registration. Useful for larger FOV_images. Larger means slower.
            'batch_size': 1000,
        },
        'DISK_LightGlue': {  ## Accuracy: Good, Speed: Fast.
            'num_features': 3000,  ## Number of features to extract and match. I've seen best results around 2048 despite higher values typically being better.
            'threshold_confidence': 0.0,  ## Higher values means fewer but better matches.
            'window_nms': 7,  ## Non-maximum suppression window size. Larger values mean fewer non-suppressed points.
        },
        'LoFTR': {  ## Accuracy: Okay. Speed: Medium.
            'model_type': 'indoor_new',
            'threshold_confidence': 0.2,  ## Higher values means fewer but better matches.
        },
        'ECC_cv2': {  ## Accuracy: Okay. Speed: Medium.
            'mode_transform': 'euclidean',  ## Must be one of {'translation', 'affine', 'euclidean', 'homography'}. See cv2 documentation on findTransformECC for more details.
            'n_iter': 200,
            'termination_eps': 1e-09,  ## Termination criteria for the registration algorithm. See documentation for more details.
            'gaussFiltSize': 1,  ## Size of the gaussian filter used to smooth the FOV_image before registration. Larger values mean more smoothing.
            'auto_fix_gaussFilt_step': 10,  ## If the registration fails, then the gaussian filter size is reduced by this amount and the registration is tried again.
        },
        'PhaseCorrelation': {  ## Accuracy: Poor. Speed: Very fast. Notes: Only applicable for translations, not rotations or scaling.
            'bandpass_freqs': [1, 30],
            'order': 5,
        },
        'NullRegistration': {},  ## No registration, no warping.
    },
    constraint='affine',  ## Must be one of {'rigid', 'euclidean', 'similarity', 'affine', 'homography'}. Choose constraint based on expected changes in images; use the simplest constraint that is applicable.
    kwargs_RANSAC = {
        'inl_thresh': 3.0,  ## cv2.findHomography RANSAC inlier threshold. Larger values mean more lenient matching.
        'max_iter': 100,
        'confidence': 0.99,
    },
    verbose=True,  ## Set to 3 to view plots of the alignment process if available for the method.
);

Plot the alignment scores. The '(final)' score is the alignment score between a pair of images given the final compsed geometric transformation between them. The '(direct)' score (only shown if the match search algorithm was used) is the alignment score between a pair of images given the single direct geometric transformation between them.

In [ ]:
aligner.plot_alignment_results_geometric();

##### 3. Fit non-rigid transformation
Play with parameters until the aligned FOV_images look good. The visualization tool below can help.

We like the following **important parameters**:
- `template`=0.5
- `template_method`='image'


In [ ]:
aligner.fit_nonrigid(
    template=0.5,  ## specifies which image to use as the template. Either array (image), integer (ims_moving index), or float (ims_moving fractional index)
    ims_moving=aligner.ims_registered_geo,  ## Input images. Typically the geometrically registered images
    remappingIdx_init=aligner.remappingIdx_geo,  ## The remappingIdx between the original images (and ROIs) and ims_moving
    template_method='image',  ## 'sequential': align images to neighboring images. 'image': align to a single image, good if using geometric registration first
    method='DeepFlow',
    kwargs_method = {
        'DeepFlow': {},  ## Accuracy: Good (good in middle, poor on edges), Speed: Fast (CPU only)
        'RoMa': {  ## Accuracy: Okay (decent in middle, poor on edges), Speed: Slow (can be fast with a GPU), Notes: This method can work on the raw images without pre-registering using geometric methods.
            'model_type': 'outdoor',
        },
        'OpticalFlowFarneback': {  ## Accuracy: Varies (can sometimes be tuned to be the best as there are no edge artifacts), Speed: Medium (CPU only)
            'pyr_scale': 0.7,
            'levels': 5,
            'winsize': 256,
            'iterations': 15,
            'poly_n': 5,
            'poly_sigma': 1.5,            
        },
        'NullRegistration': {},
    },    
)

aligner.transform_images_nonrigid(FOV_images);

In [ ]:
aligner.plot_alignment_results_nonrigid();

##### 4. Transform ROIs

In [ ]:
aligner.transform_ROIs(
    ROIs=data.spatialFootprints, 
    remappingIdx=aligner.remappingIdx_nonrigid,
    # remappingIdx=aligner.remappingIdx_geo,
    normalize=True,
);

Ensure that the aligned FOVs look aligned

In [ ]:
print(f'Pre-alignment below')
roicat.visualization.display_toggle_image_stack(data.FOV_images)
print(f'Geometric alignment below')
roicat.visualization.display_toggle_image_stack(aligner.ims_registered_geo)
print(f'Non-rigid alignment below')
roicat.visualization.display_toggle_image_stack(aligner.ims_registered_nonrigid)
print(f'Transformed ROIs below')
roicat.visualization.display_toggle_image_stack(aligner.get_ROIsAligned_maxIntensityProjection(normalize=True), clim=None)

# Blur ROIs

ROIs from different sessions with zero spatial overlap have very low probability of being considered the same ROI during the clustering step. Blurring the spatial footprint masks can increase the overlap between ROIs that drift apart from each other. It's a good idea to increase the `kernel_halfWidth` if you are working with sparsely labeled ROIs or ROIs that change/move from session to session.

In [ ]:
blurrer = roicat.tracking.blurring.ROI_Blurrer(
    frame_shape=(data.FOV_height, data.FOV_width),  ## FOV height and width
    kernel_halfWidth=4,  ## The half width of the 2D gaussian used to blur the ROI masks
    plot_kernel=False,  ## Whether to visualize the 2D gaussian
)

blurrer.blur_ROIs(
    spatialFootprints=aligner.ROIs_aligned[:],
);

See that the blurred ROIs are overlapping each other

In [ ]:
roicat.visualization.display_toggle_image_stack(blurrer.get_ROIsBlurred_maxIntensityProjection())

# ROInet embedding

This step passes the images of each ROI through the ROInet neural network. The inputs are the images, the output is an array describing the visual properties of each ROI.

Initialize the ROInet object. The `ROInet_embedder` class will automatically download and load a pretrained ROInet model. If you have a GPU, this step will be much faster.

In [ ]:
dir_temp = tempfile.gettempdir()

roinet = roicat.ROInet.ROInet_embedder(
    device=DEVICE,  ## Which torch device to use ('cpu', 'cuda', etc.)
    dir_networkFiles=dir_temp,  ## Directory to download the pretrained network to
    download_method='check_local_first',  ## Check to see if a model has already been downloaded to the location (will skip if hash matches)
    download_url='https://osf.io/x3fd2/download',  ## URL of the model
    download_hash='7a5fb8ad94b110037785a46b9463ea94',  ## Hash of the model file
    forward_pass_version='latent',  ## How the data is passed through the network
    verbose=True,  ## Whether to print updates
)

Resize ROIs and prepare a dataloader.

**Important parameters**:
- `um_per_pixel`: (same as specified in `data` object). Resolution of FOV. This is used to resize the ROIs to be relatively consistent across resolutions.

In [ ]:
data.um_per_pixel = 1.5

In [ ]:
roinet.generate_dataloader(
    ROI_images=data.ROI_images,  ## Input images of ROIs
    um_per_pixel=data.um_per_pixel,  ## Resolution of FOV
    pref_plot=False,  ## Whether or not to plot the ROI sizes
    
    jit_script_transforms=False,  ## (advanced) Whether or not to use torch.jit.script to speed things up
    
    batchSize_dataloader=8,  ## (advanced) PyTorch dataloader batch_size
    pinMemory_dataloader=True,  ## (advanced) PyTorch dataloader pin_memory
    numWorkers_dataloader=mp.cpu_count(),  ## (advanced) PyTorch dataloader num_workers
    persistentWorkers_dataloader=True,  ## (advanced) PyTorch dataloader persistent_workers
    prefetchFactor_dataloader=2,  ## (advanced) PyTorch dataloader prefetch_factor
);

In general, you want to see that a neuron fills roughly 25-50% of the area of the image.

In [ ]:
roicat.visualization.display_toggle_image_stack(roinet.ROI_images_rs[:1000], image_size=(200,200))

Pass the data through the network. Expect for large datasets (~40,000 ROIs) that this takes around 15 minutes on CPU or 1 minute on GPU.

In [ ]:
roinet.generate_latents();

# Scattering wavelet embedding

This is similar to the ROInet embedding in purpose.

In [ ]:
swt = roicat.tracking.scatteringWaveletTransformer.SWT(
    kwargs_Scattering2D={'J': 2, 'L': 12},  ## 'J' is the number of convolutional layers. 'L' is the number of wavelet angles.
    image_shape=data.ROI_images[0].shape[1:3],  ## size of a cropped ROI image
    device=DEVICE,  ## PyTorch device
)

swt.transform(
    ROI_images=roinet.ROI_images_rs,  ## All the cropped and resized ROI images
    batch_size=100,  ## Batch size for each iteration (smaller is less memory but slower)
);

# Compute similarities

Now we can compare the similarities of the ROIs. This includes calculating 4 kinds of similarities:
1. `s_sf`: 'similarity spatial footprint'. The physical overlap between ROIs.
2. `s_NN`: 'similarity neural network'. The similarities of the embeddings out of ROInet.
3. `s_SWT`: 'similarity scaterring wavelet transform'. The similarities of the embeddings out of the scattering wavelet transformer.
4. `s_sesh`: 'similarity sessions'. 0 if from the same session, 1 if from different sessions. ROIs from the same session have 0 probability of being the same.

The result of this step will be a set of pairwise similarity matrices.

Initialize the `ROI_graph` class and compute similarities.
To make computation more efficient, only ROIs within the same 'block' are compared against each other.

In [ ]:
# d_sf = helpers.sparse_neighbors_l1(
#     x_csr=blurrer.ROIs_blurred[0][:].tocsr(),
#     adjacency_matrix=None,
#     device='cpu',
#     values_dtype=torch.float32,
#     requires_grad=True,
#     accum_dtype=torch.float64,
#     return_tensor=False,
#     indices_device='cpu',
# )
# # d_sf = scipy.sparse.coo_matrix((d_sf[2].detach().cpu().numpy(), (d_sf[0].detach().cpu().numpy(), d_sf[1].detach().cpu().numpy())), shape=d_sf[3]).tocsr()

In [ ]:
%load_ext autoreload
%autoreload 2
import roicat
import roicat.helpers
import roicat.tracking.clustering

from pathlib import Path

import scipy.sparse
import torch
import numpy as np

In [ ]:
# ## temporarily save feature vectors:
# dir_save = "/Users/richardhakim/Desktop/ROICaT_temp"
# Path(dir_save).mkdir(parents=True, exist_ok=True)

# roicat.helpers.pickle_save(obj=aligner.ROIs_aligned, filepath=str(Path(dir_save) / 'ROIs_aligned.pkl'))
# roicat.helpers.pickle_save(obj=blurrer.ROIs_blurred, filepath=str(Path(dir_save) / 'ROIs_blurred.pkl'))
# roicat.helpers.pickle_save(obj=roinet.latents, filepath=str(Path(dir_save) / 'roinet_latents.pkl'))
# roicat.helpers.pickle_save(obj=swt.latents, filepath=str(Path(dir_save) / 'swt_latents.pkl'))
# roicat.helpers.pickle_save(obj=data.session_bool, filepath=str(Path(dir_save) / 'session_bool.pkl'))

In [9]:
## temporarily save feature vectors:
dir_save = "/Users/richardhakim/Desktop/ROICaT_temp"
Path(dir_save).mkdir(parents=True, exist_ok=True)

r = roicat.helpers.pickle_load(filepath=str(Path(dir_save) / 'ROIs_aligned.pkl'))
b = roicat.helpers.pickle_load(filepath=str(Path(dir_save) / 'ROIs_blurred.pkl'))
rl = roicat.helpers.pickle_load(filepath=str(Path(dir_save) / 'roinet_latents.pkl'))
sl = roicat.helpers.pickle_load(filepath=str(Path(dir_save) / 'swt_latents.pkl'))
sb = roicat.helpers.pickle_load(filepath=str(Path(dir_save) / 'session_bool.pkl'))

In [ ]:
bc = scipy.sparse.vstack(b)
bc2 = bc[:].copy().astype(np.bool_)
adj = bc2 @ bc2.T
# bc3 = roicat.helpers.scipy_sparse_to_torch_coo(sp_array=bc[:], dtype=torch.float32).to('cpu')

In [ ]:
rlst = roicat.helpers.masked_pairwise_similarity_dense(
    X=rl[:],
    adjacency=adj[:][:, :],
    metric='cosine',
    device='cpu',
    accum_dtype=torch.float32,
    return_tensor=True,
    indices_device='cpu',
    edges_per_chunk=1024,
    eps=1e-12,
).coalesce()

In [ ]:
sbis = roicat.helpers.masked_pairwise_similarity_dense(
    X=torch.as_tensor(sb),
    adjacency=adj[:][:, :],
    metric='dot',
    device='cpu',
    accum_dtype=torch.float32,
    return_tensor=False,
    indices_device='cpu',
    edges_per_chunk=1024,
    eps=1e-12,
)
# sbis
s_sesh_inv = scipy.sparse.coo_matrix((sbis[2].detach().cpu().numpy(), (sbis[0].detach().cpu().numpy(), sbis[1].detach().cpu().numpy())), shape=sbis[3]).tocsr()
s_sesh_inv.eliminate_zeros()
ssit = roicat.helpers.scipy_sparse_to_torch_coo(s_sesh_inv).coalesce()


In [ ]:
v_all = 1 - ((rlst.values() + 1)/2)
v_diff = 1 - (((rlst * ssit).coalesce().values() + 1) / 2)

In [ ]:
import matplotlib.pyplot as plt

plt.figure()
plt.hist(v_all.detach().cpu().numpy(), bins=100, label='All', alpha=0.5, density=True);
plt.hist(v_diff.detach().cpu().numpy(), bins=100, label='Different', alpha=0.5, density=True);
plt.legend();
# plt.yscale('log')

In [ ]:
# torch.nn.functional.kl_div(
#     input=v_all[np.random.randint(0, len(v_all), size=v_diff.shape[0])],
#     target=v_diff,
#     reduction='mean',
# )

In [ ]:

class SoftHistogram(torch.nn.Module):
    """
    Differentiable 1D histogram for values in a fixed range, with optional chunking.

    RH 2025

    Supports two kernels:

    1) 'sigmoid': smooth top-hats via a difference of sigmoids at bin edges.
       - Parameter `temperature` (τ) controls sharpness. A good default is
         τ ≈ (bin_width / 4).

    2) 'triangular': Parzen window with triangular basis centered at bin centers.
       - Piecewise-linear, concentrates mass similarly to a standard histogram.

    Args:
        num_bins (int):
            Number of bins over the closed interval [value_min, value_max].
        value_min (float):
            Lower bound of histogram range (inclusive).
        value_max (float):
            Upper bound of histogram range (inclusive).
        kernel (Literal['sigmoid', 'triangular']):
            Basis function for soft counts.
            - 'triangular': Parzen window with triangular basis centered at bin centers.
              Faster, but not that smooth
            - 'sigmoid': smooth top-hats via a difference of sigmoids at bin edges.
              Smooth gradients, but slower.
        temperature (Optional[float]):
            Kernel sharpness. If None, defaults to (bin_width / 4) for 'sigmoid'
            and (bin_width) for 'triangular' (the triangular kernel ignores this).
        chunk_size (Optional[int]):
            If not None, process input values in chunks of this size to bound
            peak memory. Recommended for very large N.
        oob_behavior (Literal['warn', 'raise', 'ignore']):
            Behavior when input values are outside [value_min, value_max]:
            - 'warn': issue a RuntimeWarning and clamp values in forward.
            - 'raise': raise ValueError.
            - 'ignore': do nothing (values will be clamped in forward).
        device (Optional[str]):
            Torch device for buffers. If None, inferred at runtime from input.
        dtype (torch.dtype):
            Floating dtype for internal buffers.

    Notes:
        - Returns *unnormalized* counts (sum equals number of input samples).
        - Complexity is O(N * B); use chunking to cap memory when N or B is large.
    """            
    def __init__(
        self,
        num_bins: int,
        value_min: float = 0.0,
        value_max: float = 1.0,
        *,
        kernel: Literal["sigmoid", "triangular"] = "sigmoid",
        temperature: Optional[float] = None,
        chunk_size: Optional[int] = None,
        oob_behavior: Literal["warn","raise","ignore"]="warn",
        device: Optional[str] = None,
        dtype: torch.dtype = torch.float32,
    ) -> None:
        super().__init__()
        assert num_bins >= 2, "num_bins must be >= 2"
        assert value_max > value_min, "value_max must be > value_min"
        self.num_bins = int(num_bins)
        self.value_min = float(value_min)
        self.value_max = float(value_max)
        self.kernel = kernel
        self.chunk_size = int(chunk_size) if chunk_size is not None else None
        self.oob_behavior = oob_behavior
        self.dtype = dtype

        # Static bin geometry
        edges = torch.linspace(self.value_min, self.value_max, self.num_bins + 1, dtype=dtype, device=device)
        centers = 0.5 * (edges[:-1] + edges[1:])
        bin_width = (self.value_max - self.value_min) / self.num_bins

        # Register as buffers (moved with .to(device))
        self.register_buffer("edges", edges, persistent=False)  ## persistent=False means not saved in state_dict, but still tracked by the optimizer and can be found at self.edges
        self.register_buffer("centers", centers, persistent=False)
        self.register_buffer("bin_width", torch.tensor(bin_width, dtype=dtype, device=device), persistent=False)

        # Temperature defaults
        if temperature is None:
            if self.kernel == "sigmoid":
                temperature = float(bin_width) / 4.0
            elif self.kernel == "triangular":
                temperature = float(bin_width)  # unused, placeholder for API symmetry
            else:
                raise ValueError(f"Unknown kernel: {self.kernel}")
        self.register_buffer("temperature", torch.tensor(float(temperature), dtype=dtype, device=device), persistent=False)

    @torch.no_grad()
    def _check_range(self, values: torch.Tensor) -> None:
        """
        Check if input values are within the histogram range [value_min, value_max].
        Handles out-of-bounds (OOB) values based on `oob_behavior`.
        """
        if values.isnan().any() or values.isinf().any():
            raise ValueError("Input values contain NaN or Inf.")
        eps = 1e-6
        vmin = float(values.min().item())
        vmax = float(values.max().item())
        lo_bound = self.value_min - eps
        hi_bound = self.value_max + eps

        if vmin < lo_bound or vmax > hi_bound:
            n_low = int((values < lo_bound).sum().item())
            n_high = int((values > hi_bound).sum().item())
            n = values.numel()
            msg = (f"{n_low + n_high}/{n} values out of histogram bounds [{self.value_min}, {self.value_max}] "
                   f"value extrema: (min={vmin:.6f}, max={vmax:.6f}). "
                   f"{'They will be clamped if forward(clamp=True).' if self.oob_behavior!='raise' else ''}")

            if self.oob_behavior == "raise":
                raise ValueError(msg)
            elif self.oob_behavior == "warn":
                warnings.warn(msg, RuntimeWarning)
            # elif "ignore": do nothing

    def _hist_sigmoid_chunk(self, values: torch.Tensor) -> torch.Tensor:
        """
        Soft counts via difference of sigmoids at edges (correct sign).
        Ensures non-negative bin contributions up to fp roundoff.
        Slower than triangular, but smoother gradients.
        """
        # Make CDF increasing in edge: F(e | v) = σ((e - v)/τ)
        z = (self.edges[None, :] - values[:, None]) / self.temperature  # (Nc, B+1)
        cdf_vals = torch.sigmoid(z)

        # Bin mass = F(e_{k+1}) - F(e_k) >= 0
        counts = (cdf_vals[:, 1:] - cdf_vals[:, :-1]).sum(dim=0)  ## performs a kind of recursive subtraction between neighboring bins

        # Optional: guard tiny negatives from fp error (e.g., -1e-12)
        return counts.clamp_min(0)

    def _hist_triangular_chunk(self, values: torch.Tensor) -> torch.Tensor:
        """Soft counts via triangular Parzen windows centered at bin centers (chunk)."""
        # Distance in bin-width units
        u = (values[:, None] - self.centers[None, :]) / self.bin_width  # (Nc, B); broadcasted difference
        weights = torch.clamp(1.0 - u.abs(), min=0.0)  # triangle of base 2 bin_width; non-negative values only within appropriate bins
        return weights.sum(dim=0)  # (B,)

    def forward(self, values: torch.Tensor, *, clamp: bool = True) -> torch.Tensor:
        """
        Compute differentiable histogram counts for 1-D values.

        Args:
            values (torch.Tensor):
                1-D tensor of samples, typically in [value_min, value_max].
            clamp (bool):
                If True, clamp values to [value_min, value_max] before binning.

        Returns:
            counts (torch.Tensor):
                1-D tensor of length `num_bins` with soft counts (sum ≈ len(values)).
        """
        if values.ndim != 1:
            values = values.reshape(-1)
        if clamp:
            values = values.clamp(min=self.value_min, max=self.value_max)

        # Range & device handling
        self._check_range(values)
        ## Move buffers to the same device as values
        device = values.device
        if self.edges.device != device:
            self.edges = self.edges.to(device=device, dtype=self.dtype)
            self.centers = self.centers.to(device=device, dtype=self.dtype)
            self.bin_width = self.bin_width.to(device=device, dtype=self.dtype)
            self.temperature = self.temperature.to(device=device, dtype=self.dtype)

        counts = torch.zeros(self.num_bins, device=device, dtype=self.dtype)

        # If no chunking
        if self.chunk_size is None or values.numel() <= self.chunk_size:
            if self.kernel == "sigmoid":
                counts = self._hist_sigmoid_chunk(values)
            else:
                counts = self._hist_triangular_chunk(values)
            return counts

        # Stream in chunks to keep peak memory bounded
        num = values.numel()
        for start in range(0, num, self.chunk_size):
            stop = min(start + self.chunk_size, num)
            v = values[start:stop]
            if self.kernel == "sigmoid":
                counts += self._hist_sigmoid_chunk(v)
            else:
                counts += self._hist_triangular_chunk(v)
        return counts


class DistributionSeparationLoss(torch.nn.Module):
    """
    Differentiable separation objective for 'same' vs 'different' distance distributions.

    RH 2025

    This class estimates:
        - dens_all: histogram of all pairwise distances (from d_conj.data).
        - dens_diff: histogram of intra-session (known-different) distances.
        - dens_same: softplus(dens_all - alpha * dens_diff).

    Then it minimizes:
        L = w_overlap * sum(p_same * p_diff)
          + w_margin  * relu(margin - (mean_diff - mean_same))
          + w_mass    * (mass_same - mass_target)^2
    where p_* are normalized densities over bins and means are
    expectation over bin centers.

    Args:
        num_bins (int):
            Number of bins in [0, 1] for the distance histograms.
        kernel (Literal['sigmoid', 'triangular']):
            Soft histogram kernel (passed to SoftHistogram).
        temperature (Optional[float]):
            Kernel sharpness (passed to SoftHistogram). If None, a sensible
            default based on bin width is used.
        chunk_size (Optional[int]):
            Histogram chunk size (samples) for streaming accumulation.
        scale_diff_by_ratio (bool):
            If True, scale dens_diff by len(all)/len(intra).
        sigma_bins_smooth (Optional[float]):
            Optional Gaussian smoothing (in bin units) after histogramming.
        overlap_weight (float):
            Weight for overlap penalty.
        margin_weight (float):
            Weight for mean-separation margin penalty.
        mass_weight (float):
            Weight for optional mass prior penalty.
        margin (float):
            Desired mean separation on [0, 1].
        mass_target (float):
            Prior fraction of 'same' mass among all pairs. Set weight=0 to disable.
        device (Optional[str]):
            Torch device for buffers.
        dtype (torch.dtype):
            Floating dtype.
    """

    def __init__(
        self,
        num_bins: int = 128,
        *,
        kernel: Literal["sigmoid", "triangular"] = "sigmoid",
        temperature: Optional[float] = None,
        chunk_size: Optional[int] = None,
        scale_diff_by_ratio: bool = True,
        sigma_bins_smooth: Optional[float] = None,
        overlap_weight: float = 1.0,
        margin_weight: float = 0.5,
        mass_weight: float = 0.0,
        margin: float = 0.20,
        mass_target: float = 0.05,
        device: Optional[str] = None,
        dtype: torch.dtype = torch.float32,
    ) -> None:
        super().__init__()
        self.hist = SoftHistogram(
            num_bins=num_bins,
            value_min=0.0,
            value_max=1.0,
            kernel=kernel,
            temperature=temperature,
            chunk_size=chunk_size,
            device=device,
            dtype=dtype,
        )
        self.num_bins = int(num_bins)
        self.scale_diff_by_ratio = bool(scale_diff_by_ratio)
        self.sigma_bins_smooth = sigma_bins_smooth
        self.overlap_weight = float(overlap_weight)
        self.margin_weight = float(margin_weight)
        self.mass_weight = float(mass_weight)
        self.margin = float(margin)
        self.mass_target = float(mass_target)
        self.dtype = dtype

    def _normalize_safe(self, x: torch.Tensor) -> torch.Tensor:
        eps = torch.finfo(x.dtype).eps
        s = x.sum()
        return x / (s + eps)

    def _soft_crossover(self, a: torch.Tensor, b: torch.Tensor) -> float:
        """Soft argmin over |a - b| to report a crossover distance (diagnostic)."""
        gap = (a - b).abs()
        w = torch.softmax(-gap / 1e-3, dim=0)
        centers = self.hist.centers
        return float((w * centers).sum().detach().cpu())
    
    def _gaussian_smooth1d(
        self,
        x: torch.Tensor,
        sigma_bins: Optional[float] = None,
    ) -> torch.Tensor:
        """
        Lightweight Gaussian smoothing over bins for stabilization.

        Args:
            x (torch.Tensor): 1-D vector.
            sigma_bins (Optional[float]): Standard deviation in *bin* units.
                If None or <= 0, returns x unchanged.

        Returns:
            (torch.Tensor): Smoothed 1-D vector.
        """
        if not sigma_bins or sigma_bins <= 0:
            return x
        # Kernel size ~ 6*sigma (odd)
        radius = max(1, int(math.ceil(3.0 * sigma_bins)))
        kx = torch.arange(-radius, radius + 1, device=x.device, dtype=x.dtype)
        kernel = torch.exp(-0.5 * (kx / float(sigma_bins)) ** 2)
        kernel = kernel / kernel.sum()
        x_pad = torch.nn.functional.pad(x[None, None, :], (radius, radius), mode="replicate")
        y = torch.nn.functional.conv1d(x_pad, kernel[None, None, :])[0, 0]
        return y

    def forward(
        self,
        d_conj: scipy.sparse.csr_matrix,
        s_sesh_inv: scipy.sparse.csr_matrix,
    ) -> Tuple[torch.Tensor, Dict[str, float]]:
        """
        Compute separation loss from sparse distance matrices.

        Args:
            d_conj (scipy.sparse.csr_matrix):
                Conjunctive distance matrix (values in [0, 1]).
            s_sesh_inv (scipy.sparse.csr_matrix):
                Mask for intra-session (known 'different') pairs; same shape as d_conj.

        Returns:
            (tuple):
                loss (torch.Tensor):
                    Scalar objective suitable for backprop.
                info (Dict[str, float]):
                    Diagnostics and summarized curves (detached).
        """
        # Pull values
        v_all_np = d_conj.data
        d_intra = d_conj.multiply(s_sesh_inv)
        d_intra.eliminate_zeros()
        if d_intra.nnz == 0:
            # Degenerate case: no negatives to anchor the 'different' distribution
            nan = torch.tensor(float("nan"), device=self.hist.edges.device, dtype=self.dtype)
            return nan, {"error": "no_intra_pairs"}

        v_diff_np = d_intra.data

        # Convert to tensors on the histogram's device/dtype
        device = self.hist.edges.device
        v_all = torch.as_tensor(v_all_np, device=device, dtype=self.dtype)
        v_diff = torch.as_tensor(v_diff_np, device=device, dtype=self.dtype)

        # Histograms (unnormalized counts)
        dens_all = self.hist(v_all, clamp=True)
        dens_diff = self.hist(v_diff, clamp=True)

        if self.scale_diff_by_ratio:
            scale = (v_all.numel() / max(1, v_diff.numel()))
            dens_diff = dens_diff * scale

        # Stabilize with optional smoothing (helps if counts are jagged)
        if self.sigma_bins_smooth and self.sigma_bins_smooth > 0:
            dens_all = self._gaussian_smooth1d(dens_all, sigma_bins=self.sigma_bins_smooth)
            dens_diff = self._gaussian_smooth1d(dens_diff, sigma_bins=self.sigma_bins_smooth)

        # 'Same' as positive part of (all - diff)
        dens_same = torch.nn.functional.softplus(dens_all - dens_diff)  # ≥ 0
        
        # Make a cropped version of 'same' (zero after crossover)
        idx_crossover = torch.where(dens_diff > dens_same + dens_same.max()*0.10)[0][0] + 1 if (dens_same > dens_diff).any() else self.num_bins
        d_crossover = self.hist.edges[idx_crossover].item()
        dens_same_crop = dens_same.clone()
        dens_same_crop[idx_crossover:] = 0  # zero out after crossover

        # Normalize to probability mass functions over bins
        p_same = self._normalize_safe(dens_same)
        p_diff = self._normalize_safe(dens_diff)

        # Overlap loss (encourage disjoint support)
        L_overlap = (p_same * p_diff).sum()

        # Mean-margin loss (push means apart)
        centers = self.hist.centers
        mu_same = (p_same * centers).sum()
        mu_diff = (p_diff * centers).sum()
        L_margin = torch.nn.functional.relu(self.margin - (mu_diff - mu_same))

        # Optional mass prior (fraction of 'same' among all)
        frac_same = dens_same.sum() / (dens_all.sum() + torch.finfo(dens_all.dtype).eps)
        L_mass = (frac_same - self.mass_target) ** 2

        # Total loss
        loss = (
            self.overlap_weight * L_overlap
            + self.margin_weight * L_margin
            + self.mass_weight * L_mass
        )

        # Diagnostics (detach to CPU)
        info = {
            "loss": float(loss.detach().cpu()),
            "L_overlap": float(L_overlap.detach().cpu()),
            "L_margin": float(L_margin.detach().cpu()),
            "L_mass": float(L_mass.detach().cpu()),
            "mu_same": float(mu_same.detach().cpu()),
            "mu_diff": float(mu_diff.detach().cpu()),
            "frac_same": float(frac_same.detach().cpu()),
            "soft_crossover": self._soft_crossover(dens_diff, dens_same),
            "hard_crossover": d_crossover,

            "p_same": p_same.detach().cpu().numpy(),
            "p_diff": p_diff.detach().cpu().numpy(),
            "dens_all": dens_all.detach().cpu().numpy(),
            "dens_diff": dens_diff.detach().cpu().numpy(),
            "dens_same": dens_same.detach().cpu().numpy(),
            "dens_same_crop": dens_same_crop.detach().cpu().numpy(),

            "v_all": v_all_np,
            "v_diff": v_diff_np,

            "centers": self.hist.centers.detach().cpu().numpy(),
            "edges": self.hist.edges.detach().cpu().numpy(),
        }
        # Optional: you can add the full curves if you want to log them.
        return loss, info



In [ ]:
import torch
from typing import Iterable, List, Optional, Sequence, Tuple, Union

def multiindex_argsort(
    data: torch.Tensor,
    dim: int = 0,
    keys: Optional[Sequence[int]] = None,
    descending: Union[bool, Sequence[bool]] = False,
    na_position: str = "last",
    stable: bool = True,
) -> torch.Tensor:
    """
    Compute a lexicographic sort index along one dimension using columns/rows
    from the other dimension as keys. Returns a single 1-D index tensor that
    can be applied to `data` along `dim`.

    RH 2025

    Args:
        data (torch.Tensor):
            A 2-D tensor. Typical use: sort rows (dim=0) by multiple columns
            (keys along dim=1), or sort columns (dim=1) by multiple rows.
        dim (int):
            The dimension to reorder (0 for rows, 1 for columns). (Default is 0)
        keys (Optional[Sequence[int]]):
            Indices of key vectors along the *other* dimension, ordered from
            most-significant to least-significant. If None, all keys along the
            other dimension are used in natural order (0, 1, 2, ...). For
            example, with shape (N_rows, N_cols) and dim=0, keys index columns.
        descending (Union[bool, Sequence[bool]]):
            Sort direction. Either a single bool applied to all keys, or a
            per-key iterable matching `keys` length. (Default is False)
        na_position (str):
            Where to place NaNs for floating dtypes: "last" or "first".
            (Default is "last")
        stable (bool):
            Whether to use a stable sorting routine. Leave True to guarantee
            lexicographic behavior. (Default is True)

    Returns:
        (torch.Tensor):
            indices (torch.Tensor):
                1-D int64 tensor of length `data.size(dim)` giving the order to
                apply along `dim`, e.g. `data.index_select(dim, indices)`.

    Notes:
        - This implements lexicographic ordering by performing repeated stable
          sorts: least-significant key first up to most-significant last.
        - For floating keys, NaNs are explicitly placed according to
          `na_position` regardless of PyTorch backend/device quirks.
        - Equivalent to NumPy's `lexsort` conceptually (last key is primary),
          but returns only the permutation indices for a single dimension and
          stays within PyTorch. See NumPy `lexsort` docs for the key-order rule,
          and PyTorch `argsort(stable=True)` docs for stability semantics.
    """
    # ---- Validation & shape conventions -------------------------------------------------
    if data.ndim != 2:
        raise ValueError(f"`data` must be 2-D, got shape {tuple(data.shape)}.")
    if dim not in (0, 1):
        raise ValueError(f"`dim` must be 0 or 1, got {dim}.")
    key_dim = 1 - dim
    n_to_sort = data.size(dim)
    n_keys_total = data.size(key_dim)

    # Default: use all keys along the other dimension (0..K-1), most->least significant
    if keys is None:
        keys = list(range(n_keys_total))  # type: ignore[assignment]
    else:
        keys = list(keys)

    # Normalize descending spec
    if isinstance(descending, bool):
        descending_flags: List[bool] = [descending] * len(keys)
    else:
        descending_flags = list(descending)
        if len(descending_flags) != len(keys):
            raise ValueError("`descending` must be a bool or match length of `keys`.")

    # Basic input checks
    if any((k < 0) or (k >= n_keys_total) for k in keys):
        raise IndexError(f"`keys` contains out-of-range index for key_dim={key_dim}.")

    if na_position not in ("first", "last"):
        raise ValueError("`na_position` must be 'first' or 'last'.")

    # ---- Core: repeated stable argsort from least- to most-significant key -------------
    # Start as identity permutation along the dimension to be sorted.
    indices = torch.arange(n_to_sort, device=data.device, dtype=torch.int64)

    # We sort least-significant first, so iterate in reversed key order.
    for k, desc in zip(reversed(keys), reversed(descending_flags)):
        # Extract the 1-D key vector aligned with the sorting dimension.
        # If sorting rows (dim=0): keys are columns -> data[:, k] has length n_to_sort
        # If sorting columns (dim=1): keys are rows   -> data[k, :] has length n_to_sort
        key_vector_all = data[:, k] if dim == 0 else data[k, :]

        # Reorder the key vector according to the current permutation ("previous passes").
        key_vector_cur = key_vector_all.index_select(0, indices)

        # Handle NaNs deterministically for floating dtypes.
        if torch.is_floating_point(key_vector_cur):
            nan_mask = torch.isnan(key_vector_cur)
            if nan_mask.any():
                if na_position == "last":
                    # Put NaNs at the end: replace with +/-inf depending on direction.
                    fill_value = float("-inf") if desc else float("inf")
                else:  # "first"
                    fill_value = float("inf") if desc else float("-inf")
                key_vector_cur = torch.where(
                    nan_mask,
                    torch.tensor(fill_value, device=key_vector_cur.device, dtype=key_vector_cur.dtype),
                    key_vector_cur,
                )

        # Stable argsort on this key refines the permutation.
        order = torch.argsort(key_vector_cur, descending=desc, stable=stable)  # stable merge-style ordering
        indices = indices.index_select(0, order)

    return indices

In [ ]:
from __future__ import annotations
from typing import Dict, Tuple, Optional
import torch
from torch import nn


class Wasserstein1SeparationLoss(nn.Module):
    """
    Differentiable 1-D Wasserstein-1 (Earth Mover) separation between two scalar
    distributions represented by samples `v_all` and `v_diff` in [0, 1], with small,
    optional shape regularizers.

    RH 2025

    Core idea:
        - Exact 1-D W1 equals the L1 distance between quantile functions.
        - For equal lengths, this reduces to mean(|sort(x) - sort(y)|).
        - For unequal lengths, evaluate both quantile functions on a common
          uniform grid in [0, 1] using differentiable linear interpolation over
          the sorted samples.

    Loss:
        L = -W1(v_all, v_diff)
            + lambda_diff_to_one * E_{y in v_diff}[|1 - y|]
            + lambda_bimodal_all * E_{x in v_all}[min(x, 1 - x)]

    Args:
        lambda_diff_to_one (float):
            Weight for pushing 'diff' mass toward 1 (distance to delta at 1).
        lambda_bimodal_all (float):
            Weight for pushing 'all' mass toward {0, 1} (endpoint bias).
        clamp_inputs (bool):
            If True, clamp inputs to [0, 1] before processing (stable & safe).
    """

    def __init__(
        self,
        *,
        lambda_diff_to_one: float = 0.1,
        lambda_bimodal_all: float = 0.05,
        clamp_inputs: bool = True,
    ) -> None:
        super().__init__()
        self.lambda_diff_to_one = float(lambda_diff_to_one)
        self.lambda_bimodal_all = float(lambda_bimodal_all)
        self.clamp_inputs = bool(clamp_inputs)

    @staticmethod
    def _check_vector(x: torch.Tensor, name: str) -> None:
        if not isinstance(x, torch.Tensor):
            raise TypeError(f"{name} must be a torch.Tensor.")
        if x.numel() == 0:
            raise ValueError(f"{name} must be non-empty.")
        if x.ndim != 1:
            raise ValueError(f"{name} must be a 1-D tensor; got shape {tuple(x.shape)}.")

    @staticmethod
    def _quantile_linear(sorted_values: torch.Tensor, u: torch.Tensor) -> torch.Tensor:
        """
        Piecewise-linear empirical quantile on a uniform grid.

        Uses 'type-7'-style interpolation:
            pos = u * (n - 1)
            lower = floor(pos), upper = ceil(pos)
            w = pos - lower
            Q(u) = (1 - w) * v[lower] + w * v[upper]

        Args:
            sorted_values (torch.Tensor): 1-D, ascending.
            u (torch.Tensor): 1-D grid in [0, 1].

        Returns:
            (torch.Tensor): Q(u) with shape (len(u),).
        """
        n = sorted_values.shape[0]
        if n == 1:
            # Degenerate but valid: quantile is constant
            return sorted_values.expand_as(u)

        pos = u * (n - 1)
        lower_idx = torch.floor(pos).to(torch.long)
        upper_idx = torch.clamp(lower_idx + 1, max=n - 1)

        weight_upper = pos - lower_idx.type_as(pos)        # in [0,1)
        weight_lower = 1.0 - weight_upper

        v_lower = sorted_values[lower_idx]
        v_upper = sorted_values[upper_idx]
        return weight_lower * v_lower + weight_upper * v_upper

    def _wasserstein1_grid(
        self,
        v_all: torch.Tensor,
        v_diff: torch.Tensor,
        grid_size: int,
    ) -> torch.Tensor:
        """
        Compute W1 via quantile interpolation on a shared uniform grid.

        Args:
            v_all, v_diff: 1-D tensors (not necessarily equal length), values ~ [0,1].
            grid_size: number of u-points in [0,1] for quantile evaluation.

        Returns:
            (torch.Tensor): scalar W1.
        """
        device = v_all.device
        dtype = v_all.dtype

        u = torch.linspace(0.0, 1.0, grid_size, device=device, dtype=dtype)

        x_sorted, _ = torch.sort(v_all)
        y_sorted, _ = torch.sort(v_diff)

        q_all = self._quantile_linear(x_sorted, u)   # (G,)
        q_diff = self._quantile_linear(y_sorted, u)  # (G,)

        w1 = torch.mean(torch.abs(q_all - q_diff))
        return w1

    def forward(
        self,
        v_all: torch.Tensor,
        v_diff: torch.Tensor,
        *,
        grid_size: int = 1024,
    ) -> Tuple[torch.Tensor, Dict[str, float]]:
        """
        Compute the Wasserstein-1 separation loss and diagnostics.

        Args:
            v_all (torch.Tensor):
                1-D tensor of 'all' distances; values ideally in [0, 1].
            v_diff (torch.Tensor):
                1-D tensor of 'different' (intra-session) distances; values in [0, 1].
            grid_size (int):
                Number of points for the uniform quantile grid. Must be >= 2.

        Returns:
            (tuple):
                loss (torch.Tensor):
                    Scalar objective to minimize.
                info (Dict[str, float]):
                    Diagnostics (detached) including W1 and regularizer terms.
        """
        self._check_vector(v_all, "v_all")
        self._check_vector(v_diff, "v_diff")
        if grid_size < 2:
            raise ValueError("grid_size must be >= 2.")

        # Common device/dtype and safe input domain
        if self.clamp_inputs:
            v_all = v_all.clamp(min=0.0, max=1.0)
            v_diff = v_diff.clamp(min=0.0, max=1.0)

        # Main separation: exact 1-D W1 via quantile interpolation
        w1_all_vs_diff = self._wasserstein1_grid(v_all, v_diff, grid_size=grid_size)
        loss_sep = -w1_all_vs_diff

        # Shape regularizers (small weights by default)
        reg_diff_to_one = torch.mean(torch.abs(1.0 - v_diff))                 # push 'diff' to 1
        reg_all_bimodal = torch.mean(torch.minimum(v_all, 1.0 - v_all))       # push 'all' to {0,1}

        loss = (
            loss_sep
            + self.lambda_diff_to_one * reg_diff_to_one
            + self.lambda_bimodal_all * reg_all_bimodal
        )

        # Diagnostics
        with torch.no_grad():
            mu_all = float(v_all.mean().cpu())
            mu_diff = float(v_diff.mean().cpu())
            frac_all_low = float((v_all < 0.1).float().mean().cpu())
            frac_all_high = float((v_all > 0.9).float().mean().cpu())
            frac_diff_high = float((v_diff > 0.9).float().mean().cpu())

        info = {
            "loss": float(loss.detach().cpu()),
            "W1_all_vs_diff": float(w1_all_vs_diff.detach().cpu()),
            "reg_diff_to_one": float(reg_diff_to_one.detach().cpu()),
            "reg_all_bimodal": float(reg_all_bimodal.detach().cpu()),
            "mu_all": mu_all,
            "mu_diff": mu_diff,
            "frac_all_<0.1": frac_all_low,
            "frac_all_>0.9": frac_all_high,
            "frac_diff_>0.9": frac_diff_high,
            "n_all": int(v_all.numel()),
            "n_diff": int(v_diff.numel()),
            "grid_size": int(grid_size),
        }
        return loss, info

In [ ]:
from __future__ import annotations
from typing import Dict, Optional, Tuple
import torch
import torch.nn.functional as F


class SparseContrastiveLoss(torch.nn.Module):
    """
    Contrastive learning directly on huge graphs using CSR edge lists (no torch.sparse ops).

    RH 2025

    Summary
    -------
    - We process a mini-batch of anchor rows (anchors = rows handled this step).
    - For each anchor i, we read its CSR slice once:
        cross-session neighbors -> candidates for positives
        same-session neighbors  -> negatives
    - Positive mining is fully differentiable: a row-wise softmax over cross neighbors:
          alpha_{i->j} = softmax_j(sim_{ij} / tau_soft)
      We enforce soft mutuality:
          w_ij = min(alpha_{i->j}, alpha_{j->i})
    - Loss is multi-positive NT-Xent (InfoNCE) computed from those edges.

    This avoids MKL/MPS sparse kernels entirely (common failure on macOS ARM) and
    uses only dense ops on small per-row slices, which autograd supports everywhere. 
    See PyTorch’s sparse/MKL caveats.  [MKL requirement; sparse CPU paths]  [MPS notes]
    """

    def __init__(
        self,
        *,
        temperature: float = 0.2,      # NT-Xent temperature (tau)
        tau_soft: float = 0.02,        # mining softmax temperature (smaller => peakier)
        preselect_k: Optional[int] = 64,   # optional hard cap per row before softmax
        max_anchors: int = 4096,       # how many rows to sample if anchors=None
        assume_row_sorted: bool = True, # col_idx within each row sorted; speeds mutual lookup
        eps: float = 1e-12,
    ) -> None:
        super().__init__()
        self.tau = float(temperature)
        self.tau_soft = float(tau_soft)
        self.preselect_k = int(preselect_k) if preselect_k is not None else None
        self.max_anchors = int(max_anchors)
        self.assume_row_sorted = bool(assume_row_sorted)
        self.eps = float(eps)

    # ------------------------ utilities on CSR arrays ------------------------

    @staticmethod
    def _row_slice(
        row_ptr: torch.Tensor, col_idx: torch.Tensor, vals: torch.Tensor, row: int
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """Return (cols, vals) for one row as 1-D views (no copies)."""
        start = int(row_ptr[row].item())
        end   = int(row_ptr[row + 1].item())
        return col_idx[start:end], vals[start:end]

    @staticmethod
    def _split_cross_same(same_sesh_vals_row: torch.Tensor) -> torch.Tensor:
        """Boolean mask True for cross-session edges; False for same-session."""
        return ~(same_sesh_vals_row.to(dtype=torch.bool))

    def _segment_softmax(self, x: torch.Tensor) -> torch.Tensor:
        """Stable 1-D softmax with tau_soft (sum==1)."""
        if x.numel() == 0:
            return x
        x_scaled = x / max(self.tau_soft, 1e-6)
        x_scaled = x_scaled - x_scaled.max()  # numerical stability
        w = torch.exp(x_scaled)
        return w / (w.sum() + self.eps)

    def _find_in_row(
        self, col_idx_row: torch.Tensor, vals_row: torch.Tensor, col: int
    ) -> float:
        """
        Return the value at column `col` in this row (float), or -inf if not present.
        If assume_row_sorted, uses searchsorted; else scans (slower).
        """
        if col_idx_row.numel() == 0:
            return float("-inf")
        if self.assume_row_sorted:
            pos = torch.searchsorted(col_idx_row, torch.tensor([col], device=col_idx_row.device)).item()
            if pos < col_idx_row.numel() and int(col_idx_row[pos].item()) == col:
                return float(vals_row[pos].item())
            return float("-inf")
        # fallback linear scan
        eq = (col_idx_row == col)
        if eq.any():
            return float(vals_row[eq.nonzero(as_tuple=False)[0]].item())
        return float("-inf")

    # ------------------------ main forward pass ------------------------

    def forward(
        self,
        *,
        # CSR graph (must share ordering between sim_vals and same_sesh_vals)
        row_ptr: torch.Tensor,            # (N+1,) int64
        col_idx: torch.Tensor,            # (nnz,)  int64
        sim_vals: Optional[torch.Tensor], # (nnz,)  float32/float64 OR None if computing from embeddings
        same_sesh_vals: torch.Tensor,     # (nnz,)  bool/binary {0,1}
        anchors: Optional[torch.Tensor] = None,    # (B,) int64 row indices; if None we sample
        # Optional: if provided, recompute similarities per edge so gradients reach embeddings
        embeddings: Optional[torch.Tensor] = None, # (N,D) float; if set, sim_vals is ignored
        rng: Optional[torch.Generator] = None,
    ) -> Tuple[torch.Tensor, Dict[str, float]]:
        """
        Compute multi-positive NT-Xent on a mini-batch of anchor rows.

        Args:
            row_ptr, col_idx:
                CSR structure of the sparse similarity graph.
            sim_vals:
                Edge similarities aligned with `col_idx`. If `embeddings` is provided,
                we ignore this and compute cosine(z_i, z_j) on-the-fly for visited edges.
            same_sesh_vals:
                Same CSR ordering; 1 means same-session (negative), 0 means cross-session.
            anchors:
                Indices of rows to process. If None, we sample up to `max_anchors`.
            embeddings:
                Optional embedding matrix; if set, we compute similarities per edge inside
                the loop so gradients flow to the network.
        """
        device = row_ptr.device
        N = int(row_ptr.numel() - 1)

        # Sample anchors if not provided (degree-aware sampling can be added later)
        if anchors is None:
            B = min(self.max_anchors, N)
            g = rng if rng is not None else torch.Generator(device=device)
            anchors = torch.randint(low=0, high=N, size=(B,), generator=g, device=device)

        # Convenience closures to read a row
        def get_row(i: int):
            cols_i, sims_i = self._row_slice(row_ptr, col_idx, sim_source, i)
            _, same_i = self._row_slice(row_ptr, col_idx, same_sesh_vals, i)
            return cols_i, sims_i, same_i

        # If embeddings are provided, we’ll compute similarities on the fly for visited edges;
        # otherwise we use the provided sim_vals tensor as a static source.
        if embeddings is None:
            if sim_vals is None:
                raise ValueError("Provide either sim_vals or embeddings.")
            sim_source = sim_vals
        else:
            # dummy placeholder; we will compute sims per row using embeddings
            sim_source = torch.empty_like(col_idx, dtype=embeddings.dtype, device=device)

        # Caches to avoid recomputing neighbor-row softmax multiple times
        alpha_cache: dict[int, Dict[int, float]] = {}  # row j -> {col -> alpha_j->col}

        # Accumulators for per-anchor losses
        losses = []
        anchors_with_pos = 0
        total_pos = 0
        total_neg = 0

        for i in anchors.tolist():
            # --- slice anchor row ---
            cols_i, sims_i_src = self._row_slice(row_ptr, col_idx, sim_source, i)
            _,      same_i     = self._row_slice(row_ptr, col_idx, same_sesh_vals, i)

            # Compute sims on the fly if embeddings were passed
            if embeddings is not None and cols_i.numel() > 0:
                zi = embeddings[i]
                zj = embeddings[cols_i]
                # cosine similarity; normalize once per slice
                zi_n = F.normalize(zi, dim=0)
                zj_n = F.normalize(zj, dim=1)
                sims_i = (zj_n @ zi_n)  # (deg_i,)
            else:
                sims_i = sims_i_src  # view into sim_vals

            # Partition cross-session vs same-session for this row
            cross_mask = self._split_cross_same(same_i)
            same_mask  = ~cross_mask

            pos_cols = cols_i[cross_mask]
            pos_sims = sims_i[cross_mask]
            neg_cols = cols_i[same_mask]
            neg_sims = sims_i[same_mask]

            # Optional preselect to reduce per-row work (still portable)
            if self.preselect_k is not None and pos_sims.numel() > self.preselect_k:
                v, idx = torch.topk(pos_sims, k=self.preselect_k, largest=True, sorted=False)
                pos_cols = pos_cols[idx]
                pos_sims = v

            if pos_cols.numel() == 0:
                # This anchor contributes no positives; skip it
                continue

            # Row-wise softmax mining: alpha_{i->j}
            alpha_i = self._segment_softmax(pos_sims)  # sums to 1 on this row

            # Enforce mutuality: need alpha_{j->i} per neighbor j
            w_ij_list = []
            s_pos_list = []
            for j, a_ij in zip(pos_cols.tolist(), alpha_i.tolist()):
                # compute alpha for row j if not cached
                if j not in alpha_cache:
                    cols_j, sims_j_src = self._row_slice(row_ptr, col_idx, sim_source, j)
                    _,      same_j     = self._row_slice(row_ptr, col_idx, same_sesh_vals, j)
                    
                    if embeddings is not None and cols_j.numel() > 0:
                        zj = embeddings[j]
                        zk = embeddings[cols_j]
                        zj_n = F.normalize(zj, dim=0)
                        zk_n = F.normalize(zk, dim=1)
                        sims_j = (zk_n @ zj_n)
                    else:
                        sims_j = sims_j_src

                    cross_j = self._split_cross_same(same_j)
                    cols_j_cx = cols_j[cross_j]
                    sims_j_cx = sims_j[cross_j]

                    if self.preselect_k is not None and sims_j_cx.numel() > self.preselect_k:
                        vj, idxj = torch.topk(sims_j_cx, k=self.preselect_k, largest=True, sorted=False)
                        cols_j_cx = cols_j_cx[idxj]
                        sims_j_cx = vj

                    alpha_j = self._segment_softmax(sims_j_cx)
                    alpha_cache[j] = {int(c.item()): float(a.item()) for c, a in zip(cols_j_cx, alpha_j)}

                a_ji = alpha_cache.get(j, {}).get(i, 0.0)
                w_ij = min(a_ij, a_ji)
                if w_ij > 0.0:
                    w_ij_list.append(w_ij)
                    # gather s_{ij} (we already have it)
                    s_pos_list.append(float(pos_sims[(pos_cols == j).nonzero(as_tuple=False)[0]].item()))

            if not w_ij_list:
                continue  # no mutual positives survived

            anchors_with_pos += 1
            total_pos += len(w_ij_list)
            total_neg += int(neg_sims.numel())

            # --- NT-Xent for this anchor i ---
            s_pos = torch.tensor(s_pos_list, device=device, dtype=pos_sims.dtype)
            w_pos = torch.tensor(w_ij_list, device=device, dtype=pos_sims.dtype)

            # Numerator: logsumexp over positives with log-weights
            log_num = torch.logsumexp(s_pos / self.tau + torch.log(torch.clamp(w_pos, min=self.eps)), dim=0)

            # Denominator: positives ∪ negatives (unweighted)
            if neg_sims.numel() > 0:
                den_terms = torch.cat([s_pos, neg_sims], dim=0) / self.tau
            else:
                den_terms = s_pos / self.tau
            log_den = torch.logsumexp(den_terms, dim=0)

            losses.append(-(log_num - log_den))

        if not losses:
            # No anchors had positives—return a benign zero loss (keeps graph intact)
            zero = (embeddings if embeddings is not None else row_ptr).new_tensor(0.0, requires_grad=True)
            return zero, {"anchors_with_pos": 0, "loss": 0.0}

        loss = torch.stack(losses).mean()
        info = {
            "loss": float(loss.detach().cpu()),
            "anchors": int(anchors.numel()),
            "anchors_with_pos": anchors_with_pos,
            "num_pos_pairs": total_pos,
            "num_neg_pairs": total_neg,
            "tau": self.tau,
            "tau_soft": self.tau_soft,
            "preselect_k": self.preselect_k,
        }
        return loss, info
    

import torch
from typing import Optional, Tuple

def coo_to_csr(
    row: torch.Tensor,
    col: torch.Tensor,
    values: torch.Tensor,
    *,
    n_rows: Optional[int] = None,
    n_cols: Optional[int] = None,
    coalesce: bool = True,
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Convert COO (row, col, values) to CSR (row_ptr, col_idx, values).

    RH 2025

    Args:
        row (torch.Tensor):
            1-D int tensor of length nnz with zero-based row indices.
        col (torch.Tensor):
            1-D int tensor of length nnz with zero-based column indices.
        values (torch.Tensor):
            1-D tensor of length nnz with nonzero values.
        n_rows (Optional[int]):
            Total number of rows. If None, inferred as row.max()+1 (0 if nnz==0).
        n_cols (Optional[int]):
            Total number of columns. If None, inferred as col.max()+1 (0 if nnz==0).
            Used only to form sort keys when lexicographically sorting (row, col).
        coalesce (bool):
            If True (default), combine duplicate entries with the same (row, col)
            by summing their values.

    Returns:
        (tuple):
            row_ptr (torch.Tensor):
                1-D int64 tensor of shape (n_rows+1,). row_ptr[0]=0 and
                row_ptr[r+1]-row_ptr[r] = number of nonzeros in row r.
            col_idx (torch.Tensor):
                1-D int64 tensor of column indices ordered by rows.
            values (torch.Tensor):
                1-D tensor of values aligned with col_idx.

    Notes:
        - Complexity is dominated by the sort: O(nnz log nnz).
        - The output has columns sorted within each row (because we sort by (row, col)).
        - CSR is the canonical row-compressed format used by SciPy, cuSPARSE, and PyTorch
          (`crow_indices`, `col_indices`, `values`).  [oai_citation:2‡SciPy Documentation](https://docs.scipy.org/doc/scipy/reference/sparse.html?utm_source=chatgpt.com) [oai_citation:3‡PyTorch Docs](https://docs.pytorch.org/docs/stable/sparse.html?utm_source=chatgpt.com)
    """
    # --- Validate & prepare dtypes ---
    assert row.ndim == col.ndim == values.ndim == 1, "row/col/values must be 1-D"
    assert row.numel() == col.numel() == values.numel(), "row/col/values must have same length"
    nnz = row.numel()

    # Handle degenerate empty case early
    if nnz == 0:
        n_rows = int(n_rows or 0)
        row_ptr = torch.zeros(n_rows + 1, dtype=torch.int64, device=row.device)
        col_idx = torch.empty(0, dtype=torch.int64, device=row.device)
        vals_out = values.new_empty((0,))
        return row_ptr, col_idx, vals_out

    row = row.to(dtype=torch.int64)
    col = col.to(dtype=torch.int64)

    if n_rows is None:
        n_rows = int(row.max().item()) + 1
    if n_cols is None:
        n_cols = int(col.max().item()) + 1

    # --- Lexicographic sort by (row, col) using a composite 64-bit key ---
    # key = row * (n_cols) + col
    # Assumes n_cols fits in int64 range (true for typical sparse problems).
    key = row * (n_cols if n_cols > 0 else 1) + col
    perm = torch.argsort(key, stable=True)
    row_s = row[perm]
    col_s = col[perm]
    val_s = values[perm]

    if coalesce:
        # Identify group starts where (row, col) changes
        is_start = torch.ones(nnz, dtype=torch.bool, device=row.device)
        is_start[1:] = (row_s[1:] != row_s[:-1]) | (col_s[1:] != col_s[:-1])

        # Group ids via prefix sum over starts: 0,0,0,1,1,2,...
        group_id = torch.cumsum(is_start.to(torch.int64), dim=0) - 1
        n_groups = int(group_id[-1].item()) + 1

        # Take the first occurrence of each (row, col) as representative
        rep_mask = is_start
        row_u = row_s[rep_mask]
        col_u = col_s[rep_mask]

        # Sum values within each group (segment sum)
        vals_u = torch.zeros(n_groups, dtype=val_s.dtype, device=val_s.device)
        vals_u.scatter_add_(0, group_id, val_s)

        # Build row_ptr by counting nnz per row from unique pairs
        counts = torch.bincount(row_u, minlength=n_rows)
        row_ptr = torch.empty(n_rows + 1, dtype=torch.int64, device=row.device)
        row_ptr[0] = 0
        row_ptr[1:] = torch.cumsum(counts, dim=0)

        return row_ptr, col_u, vals_u
    else:
        # No coalescing: just count per-row entries as sorted
        counts = torch.bincount(row_s, minlength=n_rows)
        row_ptr = torch.empty(n_rows + 1, dtype=torch.int64, device=row.device)
        row_ptr[0] = 0
        row_ptr[1:] = torch.cumsum(counts, dim=0)
        return row_ptr, col_s, val_s

In [ ]:
class SparseContrastiveLoss(torch.nn.Module):
    def __init__(
        self,
        *,
        temperature: float = 0.2,
        tau_soft: float = 0.02,
        preselect_k: Optional[int] = 64,
        max_negs_per_anchor: Optional[int] = None,
        assume_row_sorted: bool = True,
        mutual: bool = False,
        eps: float = 1e-12,
        auto_anchor_cap: Optional[int] = 4096,   # NEW: safeguard if anchors=None
        rng_device: Optional[str] = None,
    ) -> None:
        super().__init__()
        self.tau = float(temperature)
        self.tau_soft = float(tau_soft)
        self.preselect_k = int(preselect_k) if preselect_k is not None else None
        self.max_negs_per_anchor = int(max_negs_per_anchor) if max_negs_per_anchor is not None else None
        self.assume_row_sorted = bool(assume_row_sorted)
        self.mutual = bool(mutual)
        self.eps = float(eps)
        self.auto_anchor_cap = int(auto_anchor_cap) if auto_anchor_cap is not None else None
        self.rng_device = rng_device

    # ... helpers unchanged ...

    def forward(
        self,
        *,
        row_ptr: torch.Tensor,            # (N+1,)
        col_idx: torch.Tensor,            # (nnz,)
        sim_vals: Optional[torch.Tensor], # (nnz,) or None if embeddings given
        same_sesh_vals: torch.Tensor,     # (nnz,)
        anchors: Optional[torch.Tensor],  # (B,) or None
        embeddings: Optional[torch.Tensor] = None,
    ) -> Tuple[torch.Tensor, Dict[str, float]]:

        device = row_ptr.device
        # Ensure same_sesh_vals is boolean and aligned
        if same_sesh_vals.dtype != torch.bool:
            same_sesh_vals = same_sesh_vals != 0

        if embeddings is None and sim_vals is None:
            raise ValueError("Provide either sim_vals or embeddings.")

        # --- derive anchors if None (robust default) ---
        if anchors is None:
            # rows with degree > 0
            deg = (row_ptr[1:] - row_ptr[:-1])
            candidate_rows = torch.nonzero(deg > 0, as_tuple=False).flatten()
            if candidate_rows.numel() == 0:
                zero = (embeddings if embeddings is not None else row_ptr).new_tensor(0.0, requires_grad=True)
                return zero, {"anchors_with_pos": 0, "loss": 0.0}
            if self.auto_anchor_cap is not None and candidate_rows.numel() > self.auto_anchor_cap:
                g = torch.Generator(device=device if (device.type != "cpu") else None)
                if self.rng_device:
                    g = torch.Generator(device=self.rng_device)
                idx = torch.randint(0, candidate_rows.numel(), (self.auto_anchor_cap,), generator=g, device=device)
                anchors = candidate_rows[idx]
            else:
                anchors = candidate_rows
        # --- end derive anchors ---

        dtype_vals = (embeddings.dtype if embeddings is not None else sim_vals.dtype)

        pos_anchor = []
        pos_neighbor = []
        pos_sim = []
        neg_anchor = []
        neg_sim = []

        row_cache: dict[int, Tuple[torch.Tensor, torch.Tensor, torch.Tensor]] = {}

        for i in anchors.tolist():
            # views into CSR arrays
            cols_i_start = int(row_ptr[i].item())
            cols_i_end   = int(row_ptr[i+1].item())
            cols_i = col_idx[cols_i_start:cols_i_end]
            same_i = same_sesh_vals[cols_i_start:cols_i_end]

            # compute sims on the fly if embeddings provided, else use sim_vals slice
            if embeddings is not None and cols_i.numel() > 0:
                zi = F.normalize(embeddings[i], dim=0)
                zj = F.normalize(embeddings[cols_i], dim=1)
                sims_i = (zj @ zi)
            else:
                sims_i = sim_vals[cols_i_start:cols_i_end].to(dtype=dtype_vals, device=device)

            cross_mask = ~(same_i)
            same_mask  = same_i

            cols_cx = cols_i[cross_mask]
            sims_cx = sims_i[cross_mask]

            if self.preselect_k is not None and sims_cx.numel() > self.preselect_k:
                v, idx = torch.topk(sims_cx, k=self.preselect_k, largest=True, sorted=False)
                cols_cx, sims_cx = cols_cx[idx], v

            if cols_cx.numel() > 0:
                pos_anchor.append(torch.full_like(cols_cx, i))
                pos_neighbor.append(cols_cx)
                pos_sim.append(sims_cx)

            sims_neg = sims_i[same_mask]
            if self.max_negs_per_anchor is not None and sims_neg.numel() > self.max_negs_per_anchor:
                vneg, ineg = torch.topk(sims_neg, k=self.max_negs_per_anchor, largest=True, sorted=False)
                sims_neg = vneg
            if sims_neg.numel() > 0:
                neg_anchor.append(torch.full((sims_neg.numel(),), i, dtype=torch.long, device=device))
                neg_sim.append(sims_neg)

            row_cache[i] = (cols_i, sims_i, same_i)

        if not pos_anchor:
            zero = (embeddings if embeddings is not None else row_ptr).new_tensor(0.0, requires_grad=True)
            return zero, {"anchors_with_pos": 0, "loss": 0.0}

        pos_anchor = torch.cat(pos_anchor)
        pos_neighbor = torch.cat(pos_neighbor)
        pos_sim = torch.cat(pos_sim).to(dtype=dtype_vals, device=device)

        # per-anchor softmax α_{i->j}
        unique_anchors, seg_ids = torch.unique(pos_anchor, sorted=True, return_inverse=True)
        alpha_ij = torch.empty_like(pos_sim)
        for a_idx in range(unique_anchors.numel()):
            mask = (seg_ids == a_idx)
            sims = pos_sim[mask]
            sims_scaled = sims / max(self.tau_soft, 1e-6)
            sims_scaled = sims_scaled - sims_scaled.max()
            w = torch.exp(sims_scaled)
            alpha_ij[mask] = w / (w.sum() + self.eps)

        # mutual-softmax (optional)
        if self.mutual:
            uniq_nbrs, inv_n = torch.unique(pos_neighbor, sorted=False, return_inverse=True)
            alpha_ji_for_p = torch.zeros_like(pos_sim)
            for j_group in range(uniq_nbrs.numel()):
                j = int(uniq_nbrs[j_group].item())
                if j in row_cache:
                    cols_j, sims_j, same_j = row_cache[j]
                else:
                    start = int(row_ptr[j].item()); end = int(row_ptr[j+1].item())
                    cols_j = col_idx[start:end]
                    same_j = same_sesh_vals[start:end]
                    if embeddings is not None and cols_j.numel() > 0:
                        zj = F.normalize(embeddings[j], dim=0)
                        zk = F.normalize(embeddings[cols_j], dim=1)
                        sims_j = (zk @ zj)
                    else:
                        sims_j = sim_vals[start:end].to(dtype=dtype_vals, device=device)

                cross_j = ~(same_j)
                cols_j_cx = cols_j[cross_j]
                sims_j_cx = sims_j[cross_j]
                if self.preselect_k is not None and sims_j_cx.numel() > self.preselect_k:
                    vj, idxj = torch.topk(sims_j_cx, k=self.preselect_k, largest=True, sorted=False)
                    cols_j_cx, sims_j_cx = cols_j_cx[idxj], vj
                if cols_j_cx.numel() > 1:
                    ord_j = torch.argsort(cols_j_cx)
                    cols_j_cx = cols_j_cx[ord_j]; sims_j_cx = sims_j_cx[ord_j]
                # softmax for row j
                s = sims_j_cx / max(self.tau_soft, 1e-6)
                s = s - s.max()
                wj = torch.exp(s); wj = wj / (wj.sum() + self.eps)

                # fill α_{j->i} for all (i,j) in this group
                mask_p = (inv_n == j_group)
                idx_p = mask_p.nonzero(as_tuple=False).squeeze(1)
                i_for_j = pos_anchor[idx_p]
                # searchsorted per i
                # (col_idx rows are sorted by construction; searched row j slice re-sorted above)
                for p_idx, i_node in zip(idx_p.tolist(), i_for_j.tolist()):
                    # lower_bound search
                    lb = torch.searchsorted(cols_j_cx, torch.tensor([i_node], device=cols_j_cx.device)).item()
                    if lb < cols_j_cx.numel() and int(cols_j_cx[lb]) == int(i_node):
                        alpha_ji_for_p[p_idx] = wj[lb]
            alpha_ij = torch.minimum(alpha_ij, alpha_ji_for_p)
            keep = alpha_ij > 0
            if not torch.any(keep):
                zero = (embeddings if embeddings is not None else row_ptr).new_tensor(0.0, requires_grad=True)
                return zero, {"anchors_with_pos": 0, "loss": 0.0}
            pos_anchor = pos_anchor[keep]; pos_neighbor = pos_neighbor[keep]
            pos_sim = pos_sim[keep]; alpha_ij = alpha_ij[keep]

        # negatives
        if neg_anchor:
            neg_anchor = torch.cat(neg_anchor)
            neg_sim = torch.cat(neg_sim).to(dtype=dtype_vals, device=device)
        else:
            neg_anchor = pos_anchor.new_empty((0,))
            neg_sim = pos_sim.new_empty((0,))

        # per-anchor NT-Xent
        loss_terms = []
        anchors_with_pos = 0
        for i in anchors.tolist():
            mask_pos = (pos_anchor == i)
            if not torch.any(mask_pos):
                continue
            anchors_with_pos += 1
            s_pos = pos_sim[mask_pos] / self.tau
            w_pos = torch.clamp(alpha_ij[mask_pos], min=self.eps)
            log_num = torch.logsumexp(s_pos + torch.log(w_pos), dim=0)

            s_den_list = [s_pos]
            mask_neg = (neg_anchor == i)
            if torch.any(mask_neg):
                s_den_list.append(neg_sim[mask_neg] / self.tau)
            log_den = torch.logsumexp(torch.cat(s_den_list), dim=0)

            loss_terms.append(-(log_num - log_den))

        if not loss_terms:
            zero = (embeddings if embeddings is not None else row_ptr).new_tensor(0.0, requires_grad=True)
            return zero, {"anchors_with_pos": 0, "loss": 0.0}

        loss = torch.stack(loss_terms).mean()
        info = {
            "loss": float(loss.detach().cpu()),
            "anchors": int(anchors.numel()),
            "anchors_with_pos": anchors_with_pos,
            "num_pos_pairs": int(pos_anchor.numel()),
            "num_neg_pairs": int(neg_anchor.numel()),
            "mutual": self.mutual,
            "tau": self.tau,
            "tau_soft": self.tau_soft,
            "preselect_k": self.preselect_k,
            "max_negs_per_anchor": self.max_negs_per_anchor,
        }
        return loss, info

In [ ]:
from __future__ import annotations
from dataclasses import dataclass
from typing import Dict, Optional, Tuple, Union
import torch
import torch.nn.functional as F

# ------------------------------- Simple Sparse Holders -------------------------------

@dataclass
class SimpleCOO:
    """
    Minimal COO container.

    Args:
        row (torch.Tensor): (nnz,) int64 row indices.
        col (torch.Tensor): (nnz,) int64 col indices.
        val (torch.Tensor): (nnz,) values (float/bool).
        shape (Tuple[int, int]): matrix shape (N, M).

    Notes:
        - Indices should be 0-based and within shape.
        - Expected to be sorted lexicographically by (row, col) and coalesced.
    """
    row: torch.Tensor
    col: torch.Tensor
    val: torch.Tensor
    shape: Tuple[int, int]

    def to_csr(self) -> "SimpleCSR":
        """Convert COO -> CSR (rows sorted, duplicates summed)."""
        N, M = self.shape
        if self.row.numel() == 0:
            row_ptr = torch.zeros(N + 1, dtype=torch.int64, device=self.row.device)
            return SimpleCSR(row_ptr=row_ptr, col_idx=self.col.new_empty((0,), dtype=torch.int64),
                             val=self.val.new_empty((0,)), shape=self.shape)

        # Ensure int64
        row = self.row.to(torch.int64)
        col = self.col.to(torch.int64)
        val = self.val

        # Lexicographic sort by (row, col)
        key = row * (M if M > 0 else 1) + col
        perm = torch.argsort(key, stable=True)
        row_s, col_s, val_s = row[perm], col[perm], val[perm]

        # Coalesce duplicates: detect group starts where (row,col) changes
        is_start = torch.ones_like(row_s, dtype=torch.bool)
        is_start[1:] = (row_s[1:] != row_s[:-1]) | (col_s[1:] != col_s[:-1])
        group_id = torch.cumsum(is_start.to(torch.int64), dim=0) - 1
        n_groups = int(group_id[-1].item()) + 1

        row_u = row_s[is_start]
        col_u = col_s[is_start]
        val_u = torch.zeros(n_groups, dtype=val_s.dtype, device=val_s.device)
        val_u.scatter_add_(0, group_id, val_s)

        # Build row_ptr counts from coalesced rows
        counts = torch.bincount(row_u, minlength=N)
        row_ptr = torch.empty(N + 1, dtype=torch.int64, device=row.device)
        row_ptr[0] = 0
        row_ptr[1:] = torch.cumsum(counts, dim=0)

        return SimpleCSR(row_ptr=row_ptr, col_idx=col_u, val=val_u, shape=self.shape)

    # SciPy bridges (optional)
    def to_scipy(self):
        """Return scipy.sparse.coo_matrix (requires SciPy)."""
        import scipy.sparse as sp
        return sp.coo_matrix((self.val.detach().cpu().numpy(),
                              (self.row.detach().cpu().numpy(), self.col.detach().cpu().numpy())),
                             shape=self.shape)

    @staticmethod
    def from_scipy(A) -> "SimpleCOO":
        """Build from scipy.sparse.coo_matrix or csr_matrix."""
        import scipy.sparse as sp
        if sp.isspmatrix_csr(A):
            A = A.tocoo()
        assert sp.isspmatrix_coo(A)
        row = torch.tensor(A.row, dtype=torch.int64)
        col = torch.tensor(A.col, dtype=torch.int64)
        val = torch.tensor(A.data)
        return SimpleCOO(row=row, col=col, val=val, shape=A.shape)

    def sort_indices_(self):
        """Sort indices in-place (CSR format)."""
        idx = multiindex_argsort(
            data=torch.stack((self.row, self.col), dim=0),
            dim=1,
        )
        self.row = self.row[idx]
        self.col = self.col[idx]
        self.val = self.val[idx]


@dataclass
class SimpleCSR:
    """
    Minimal CSR container.

    Args:
        row_ptr (torch.Tensor): (N+1,) int64 row pointer (crow_indices).
        col_idx (torch.Tensor): (nnz,) int64 column indices (sorted per row).
        val (torch.Tensor): (nnz,) values (float/bool).
        shape (Tuple[int, int]): matrix shape (N, M).
    """
    row_ptr: torch.Tensor
    col_idx: torch.Tensor
    val: torch.Tensor
    shape: Tuple[int, int]

    def to_coo(self) -> SimpleCOO:
        """Convert CSR -> COO (preserves order)."""
        N, _ = self.shape
        deg = self.row_ptr[1:] - self.row_ptr[:-1]               # (N,)
        row = torch.repeat_interleave(torch.arange(N, device=self.row_ptr.device, dtype=torch.int64), deg)
        return SimpleCOO(row=row, col=self.col_idx, val=self.val, shape=self.shape)

    def to_scipy(self):
        """Return scipy.sparse.csr_matrix (requires SciPy)."""
        import scipy.sparse as sp
        N, M = self.shape
        return sp.csr_matrix((self.val.detach().cpu().numpy(),
                              self.col_idx.detach().cpu().numpy(),
                              self.row_ptr.detach().cpu().numpy()),
                             shape=(N, M))

    @staticmethod
    def from_scipy(A) -> "SimpleCSR":
        """Build from scipy.sparse.csr_matrix or coo_matrix."""
        import scipy.sparse as sp
        if sp.isspmatrix_coo(A):
            A = A.tocsr()
        assert sp.isspmatrix_csr(A)
        row_ptr = torch.tensor(A.indptr, dtype=torch.int64)
        col_idx = torch.tensor(A.indices, dtype=torch.int64)
        val = torch.tensor(A.data)
        return SimpleCSR(row_ptr=row_ptr, col_idx=col_idx, val=val, shape=A.shape)


In [ ]:

# ------------------------------- Utility: segmented reductions -------------------------------

def _segment_amax(index: torch.Tensor, src: torch.Tensor, num_segments: int, init: float = -float("inf")) -> torch.Tensor:
    """
    Per-segment max using scatter_reduce (vectorized).

    Args:
        index: (E,) int64 segment ids (e.g., row ids per edge).
        src:   (E,) values.
        num_segments: number of segments (e.g., N rows).
        init:  initial fill value.

    Returns:
        out: (num_segments,) maximum per segment (init if empty).
    """
    out = torch.full((num_segments,), init, dtype=src.dtype, device=src.device)
    # PyTorch scatter_reduce: out.scatter_reduce_(dim, index, src, reduce='amax', include_self=True)
    out.scatter_reduce_(0, index, src, reduce="amax", include_self=True)  # keep init for empty segments
    return out


def _segment_sum(index: torch.Tensor, src: torch.Tensor, num_segments: int) -> torch.Tensor:
    """
    Per-segment sum using scatter_add (vectorized).
    """
    out = torch.zeros((num_segments,), dtype=src.dtype, device=src.device)
    out.scatter_add_(0, index, src)
    return out


# ------------------------------- Core loss: mutual-softmax NT-Xent -------------------------------

class MutualSoftmaxNTXentSparse(torch.nn.Module):
    """
    Multi-positive NT-Xent on a sparse similarity graph (CSR/COO), with mutual-softmax mining.

    RH 2025

    Overview
    --------
    - Inputs: `sim` (cosine similarity) and `s_sesh` (same-session mask), same sparsity pattern.
    - Positives = cross-session edges, weighted by *mutual softmax*:
          α_{i→j} = softmax_j( sim[i,j] / tau_soft ) over j in cross-neighbors of row i
          w_{ij} = min( α_{i→j}, α_{j→i} )  (0 if reverse edge missing)
    - Negatives = same-session edges.
    - Objective (per row i with at least one positive):
          L_i = -log ( Σ_{j∈P(i)} w_{ij} e^{sim[i,j]/τ} / Σ_{q∈P(i)∪N(i)} e^{sim[i,q]/τ} )
      L = mean_i L_i.

    Args:
        temperature (float): τ in NT-Xent (default 0.2).
        tau_soft (float): temperature for mining softmax α (default 0.02).
        require_mutual (bool): if True, use min(α_{i→j}, α_{j→i}); else use α_{i→j} only.
        eps (float): numerical epsilon.

    Notes:
        - Fully vectorized: no Python loops over rows; uses segmented max/sum via scatter ops.
        - Works best if rows are column-sorted and coalesced (standard CSR).  [oai_citation:5‡PyTorch](https://pytorch.org/docs/stable/generated/torch.sparse_csr_tensor.html?utm_source=chatgpt.com) [oai_citation:6‡PyTorch Documentation](https://docs.pytorch.org/docs/stable/sparse.html?utm_source=chatgpt.com)
        - InfoNCE/NT-Xent formulation per SimCLR (multi-positive generalization).  [oai_citation:7‡Proceedings of Machine Learning Research](https://proceedings.mlr.press/v119/chen20j/chen20j.pdf?utm_source=chatgpt.com) [oai_citation:8‡NeurIPS Proceedings](https://proceedings.neurips.cc/paper/2020/file/d89a66c7c80a29b1bdbab0f2a1a94af8-Paper.pdf?utm_source=chatgpt.com)
    """
    def __init__(self, *, temperature: float = 0.2, tau_soft: float = 0.02, require_mutual: bool = True, eps: float = 1e-12) -> None:
        super().__init__()
        self.tau = float(temperature)
        self.tau_soft = float(tau_soft)
        self.require_mutual = bool(require_mutual)
        self.eps = float(eps)

    @torch.no_grad()
    def _ensure_csr(self, A: Union[SimpleCSR, SimpleCOO]) -> SimpleCSR:
        return A if isinstance(A, SimpleCSR) else A.to_csr()

    def forward(
        self,
        sim: Union[SimpleCSR, SimpleCOO],
        s_sesh: Union[SimpleCSR, SimpleCOO],
    ) -> Tuple[torch.Tensor, Dict[str, float]]:
        """
        Compute mutual-softmax NT-Xent from sparse similarities.

        Args:
            sim:    Sparse similarities (cosine-ish, may include small negatives).
            s_sesh: Same sparsity; boolean/binary values (1 = same-session).

        Returns:
            (loss, info)
        """
        sim_csr = self._ensure_csr(sim)
        ss_csr  = self._ensure_csr(s_sesh)
        assert sim_csr.shape == ss_csr.shape, "sim and s_sesh shapes must match"
        # Hard requirement in your pipeline: exact same (row_ptr, col_idx) layout
        if not (torch.equal(sim_csr.row_ptr, ss_csr.row_ptr) and torch.equal(sim_csr.col_idx, ss_csr.col_idx)):
            raise ValueError("sim and s_sesh must share identical CSR structure (row_ptr & col_idx).")

        row_ptr, col_idx, svals, ssv = sim_csr.row_ptr, sim_csr.col_idx, sim_csr.val, ss_csr.val
        N, M = sim_csr.shape
        assert N == M, "Expect square similarity matrix."

        device = svals.device
        dtype  = svals.dtype
        nnz    = col_idx.numel()

        # Build row indices for all edges: row_e ∈ [0..N-1] per edge e
        deg = row_ptr[1:] - row_ptr[:-1]                          # (N,)
        row_e = torch.repeat_interleave(torch.arange(N, device=device, dtype=torch.int64), deg)  # (nnz,)

        # Partition edges
        same_mask  = (ssv.to(torch.bool))                         # same-session -> negatives
        cross_mask = ~same_mask                                   # cross-session -> positive candidates

        row_pos = row_e[cross_mask]             # (E_pos,)
        col_pos = col_idx[cross_mask]           # (E_pos,)
        sim_pos = svals[cross_mask].to(dtype)   # (E_pos,)

        row_neg = row_e[same_mask]              # (E_neg,)
        sim_neg = svals[same_mask].to(dtype)    # (E_neg,)

        # --- Row-wise softmax over cross neighbors: α_{i→j} ---
        # α_{i→j} = softmax over edges with same row i
        # stable: subtract per-row max then exponentiate
        sim_pos_scaled = sim_pos / max(self.tau_soft, 1e-6)
        max_pos_per_row = _segment_amax(row_pos, sim_pos_scaled, num_segments=N, init=-float("inf"))
        sim_pos_centered = sim_pos_scaled - max_pos_per_row[row_pos]
        exp_pos = torch.exp(sim_pos_centered)
        sum_exp_per_row = _segment_sum(row_pos, exp_pos, num_segments=N) + self.eps
        alpha_forward = exp_pos / sum_exp_per_row[row_pos]        # (E_pos,)

        # --- (Optional) mutuality: need α_{j→i} on reverse edge if it exists ---
        if self.require_mutual:
            # Build vectorized reverse lookup via integer keys
            # key(e)   = row* N + col;  rev_key(e) = col* N + row
            N64 = torch.tensor(N, dtype=torch.int64, device=device)
            keys = row_pos * N64 + col_pos
            rev_keys = col_pos * N64 + row_pos

            order = torch.argsort(keys, stable=True)
            keys_sorted = keys[order]
            # search positions of rev_keys in keys_sorted
            pos = torch.searchsorted(keys_sorted, rev_keys)
            # verify matches
            match = (pos < keys_sorted.numel()) & (keys_sorted[pos] == rev_keys)
            alpha_rev = torch.zeros_like(alpha_forward)
            # gather α on matched reverse edges
            alpha_rev[match] = alpha_forward[order[pos[match]]]
            w_pos = torch.minimum(alpha_forward, alpha_rev)       # mutual weight
        else:
            w_pos = alpha_forward

        # Keep only edges with positive weight
        keep = w_pos > 0
        if not torch.any(keep):
            zero = svals.new_tensor(0.0, requires_grad=True)
            return zero, {"anchors_with_pos": 0, "loss": 0.0, "E_pos": 0, "E_neg": int(sim_neg.numel())}

        row_pos = row_pos[keep]
        sim_pos = sim_pos[keep]
        w_pos   = w_pos[keep]

        # --- Per-row NT-Xent: vectorized log-sum-exp with segmented reductions ---
        # Numerator per row i: log Σ_{j∈P(i)} w_ij * exp(sim_ij/τ)
        s_pos_tau = sim_pos / self.tau
        # stable: center by per-row max over positives
        max_pos_tau = _segment_amax(row_pos, s_pos_tau, num_segments=N, init=-float("inf"))
        exp_num = torch.exp(s_pos_tau - max_pos_tau[row_pos]) * torch.clamp(w_pos, min=self.eps)
        sum_num = _segment_sum(row_pos, exp_num, num_segments=N)
        log_num = torch.log(sum_num + self.eps) + max_pos_tau

        # Denominator per row i: log Σ_{q∈P(i)∪N(i)} exp(sim_iq/τ)
        s_neg_tau = sim_neg / self.tau if sim_neg.numel() else sim_neg
        # need max over union: max( max_pos_tau, max_neg_tau )
        if sim_neg.numel():
            max_neg_tau = _segment_amax(row_neg, s_neg_tau, num_segments=N, init=-float("inf"))
            denom_max = torch.maximum(max_pos_tau, max_neg_tau)
        else:
            denom_max = max_pos_tau

        # sum exp over positives (centered by denom_max)
        exp_pos_den = torch.exp(s_pos_tau - denom_max[row_pos])
        sum_pos_den = _segment_sum(row_pos, exp_pos_den, num_segments=N)

        # sum exp over negatives
        if sim_neg.numel():
            exp_neg_den = torch.exp(s_neg_tau - denom_max[row_neg])
            sum_neg_den = _segment_sum(row_neg, exp_neg_den, num_segments=N)
            sum_den = sum_pos_den + sum_neg_den
        else:
            sum_den = sum_pos_den
        log_den = torch.log(sum_den + self.eps) + denom_max

        # Consider only rows that have at least one positive
        count_pos = torch.bincount(row_pos, minlength=N)
        valid = count_pos > 0
        if not torch.any(valid):
            zero = svals.new_tensor(0.0, requires_grad=True)
            return zero, {"anchors_with_pos": 0, "loss": 0.0, "E_pos": int(row_pos.numel()), "E_neg": int(sim_neg.numel())}

        per_row = -(log_num[valid] - log_den[valid])
        loss = per_row.mean()

        info = {
            "loss": float(loss.detach().cpu()),
            "rows_with_pos": int(valid.sum().item()),
            "E_pos": int(row_pos.numel()),
            "E_neg": int(sim_neg.numel()),
            "temperature": self.tau,
            "tau_soft": self.tau_soft,
            "mutual": self.require_mutual,
        }
        return loss, info

In [ ]:

# ------------------------------- Utility: segmented reductions -------------------------------

def _segment_amax(index: torch.Tensor, src: torch.Tensor, num_segments: int, init: float = -float("inf")) -> torch.Tensor:
    """
    Per-segment max using scatter_reduce (vectorized).

    Args:
        index: (E,) int64 segment ids (e.g., row ids per edge).
        src:   (E,) values.
        num_segments: number of segments (e.g., N rows).
        init:  initial fill value.

    Returns:
        out: (num_segments,) maximum per segment (init if empty).
    """
    out = torch.full((num_segments,), init, dtype=src.dtype, device=src.device)
    # PyTorch scatter_reduce: out.scatter_reduce_(dim, index, src, reduce='amax', include_self=True)
    out.scatter_reduce_(0, index, src, reduce="amax", include_self=True)  # keep init for empty segments
    return out


def _segment_sum(index: torch.Tensor, src: torch.Tensor, num_segments: int) -> torch.Tensor:
    """
    Per-segment sum using scatter_add (vectorized).
    """
    out = torch.zeros((num_segments,), dtype=src.dtype, device=src.device)
    out.scatter_add_(0, index, src)
    return out


# ------------------------------- Core loss: mutual-softmax NT-Xent -------------------------------

class MutualSoftmaxNTXentSparse(torch.nn.Module):
    """
    Multi-positive NT-Xent on a sparse similarity graph (CSR/COO), with mutual-softmax mining.

    RH 2025

    Overview
    --------
    - Inputs: `sim` (cosine similarity) and `s_sesh` (same-session mask), same sparsity pattern.
    - Positives = cross-session edges, weighted by *mutual softmax*:
          α_{i→j} = softmax_j( sim[i,j] / tau_soft ) over j in cross-neighbors of row i
          w_{ij} = min( α_{i→j}, α_{j→i} )  (0 if reverse edge missing)
    - Negatives = same-session edges.
    - Objective (per row i with at least one positive):
          L_i = -log ( Σ_{j∈P(i)} w_{ij} e^{sim[i,j]/τ} / Σ_{q∈P(i)∪N(i)} e^{sim[i,q]/τ} )
      L = mean_i L_i.

    Args:
        temperature (float): τ in NT-Xent (default 0.2).
        tau_soft (float): temperature for mining softmax α (default 0.02).
        require_mutual (bool): if True, use min(α_{i→j}, α_{j→i}); else use α_{i→j} only.
        eps (float): numerical epsilon.

    Notes:
        - Fully vectorized: no Python loops over rows; uses segmented max/sum via scatter ops.
        - Works best if rows are column-sorted and coalesced (standard CSR).  [oai_citation:5‡PyTorch](https://pytorch.org/docs/stable/generated/torch.sparse_csr_tensor.html?utm_source=chatgpt.com) [oai_citation:6‡PyTorch Documentation](https://docs.pytorch.org/docs/stable/sparse.html?utm_source=chatgpt.com)
        - InfoNCE/NT-Xent formulation per SimCLR (multi-positive generalization).  [oai_citation:7‡Proceedings of Machine Learning Research](https://proceedings.mlr.press/v119/chen20j/chen20j.pdf?utm_source=chatgpt.com) [oai_citation:8‡NeurIPS Proceedings](https://proceedings.neurips.cc/paper/2020/file/d89a66c7c80a29b1bdbab0f2a1a94af8-Paper.pdf?utm_source=chatgpt.com)
    """
    def __init__(self, *, temperature: float = 0.2, tau_soft: float = 0.02, require_mutual: bool = True, eps: float = 1e-12) -> None:
        super().__init__()
        self.tau = float(temperature)
        self.tau_soft = float(tau_soft)
        self.require_mutual = bool(require_mutual)
        self.eps = float(eps)

    @torch.no_grad()
    def _ensure_csr(self, A: Union[SimpleCSR, SimpleCOO]) -> SimpleCSR:
        return A if isinstance(A, SimpleCSR) else A.to_csr()

    def forward(
        self,
        sim: Union[SimpleCSR, SimpleCOO],
        s_sesh: Union[SimpleCSR, SimpleCOO],
    ) -> Tuple[torch.Tensor, Dict[str, float]]:
        """
        Compute mutual-softmax NT-Xent from sparse similarities.

        Args:
            sim:    Sparse similarities (cosine-ish, may include small negatives).
            s_sesh: Same sparsity; boolean/binary values (1 = same-session).

        Returns:
            (loss, info)
        """
        sim_csr = self._ensure_csr(sim)
        ss_csr  = self._ensure_csr(s_sesh)
        assert sim_csr.shape == ss_csr.shape, "sim and s_sesh shapes must match"
        # Hard requirement in your pipeline: exact same (row_ptr, col_idx) layout
        if not (torch.equal(sim_csr.row_ptr, ss_csr.row_ptr) and torch.equal(sim_csr.col_idx, ss_csr.col_idx)):
            raise ValueError("sim and s_sesh must share identical CSR structure (row_ptr & col_idx).")

        row_ptr, col_idx, svals, ssv = sim_csr.row_ptr, sim_csr.col_idx, sim_csr.val, ss_csr.val
        N, M = sim_csr.shape
        assert N == M, "Expect square similarity matrix."

        device = svals.device
        dtype  = svals.dtype
        nnz    = col_idx.numel()

        # Build row indices for all edges: row_e ∈ [0..N-1] per edge e
        deg = row_ptr[1:] - row_ptr[:-1]                          # (N,)
        row_e = torch.repeat_interleave(torch.arange(N, device=device, dtype=torch.int64), deg)  # (nnz,)

        # Partition edges
        same_mask  = (ssv.to(torch.bool))                         # same-session -> negatives
        cross_mask = ~same_mask                                   # cross-session -> positive candidates

        row_pos = row_e[cross_mask]             # (E_pos,)
        col_pos = col_idx[cross_mask]           # (E_pos,)
        sim_pos = svals[cross_mask].to(dtype)   # (E_pos,)

        row_neg = row_e[same_mask]              # (E_neg,)
        sim_neg = svals[same_mask].to(dtype)    # (E_neg,)

        # --- Row-wise softmax over cross neighbors: α_{i→j} ---
        # α_{i→j} = softmax over edges with same row i
        # stable: subtract per-row max then exponentiate
        sim_pos_scaled = sim_pos / max(self.tau_soft, 1e-6)
        max_pos_per_row = _segment_amax(row_pos, sim_pos_scaled, num_segments=N, init=-float("inf"))
        sim_pos_centered = sim_pos_scaled - max_pos_per_row[row_pos]
        exp_pos = torch.exp(sim_pos_centered)
        sum_exp_per_row = _segment_sum(row_pos, exp_pos, num_segments=N) + self.eps
        alpha_forward = exp_pos / sum_exp_per_row[row_pos]        # (E_pos,)

        # --- (Optional) mutuality: need α_{j→i} on reverse edge if it exists ---
        if self.require_mutual:
            # Build vectorized reverse lookup via integer keys
            # key(e)   = row* N + col;  rev_key(e) = col* N + row
            N64 = torch.tensor(N, dtype=torch.int64, device=device)
            keys = row_pos * N64 + col_pos
            rev_keys = col_pos * N64 + row_pos

            order = torch.argsort(keys, stable=True)
            keys_sorted = keys[order]
            # search positions of rev_keys in keys_sorted
            pos = torch.searchsorted(keys_sorted, rev_keys)
            # verify matches
            match = (pos < keys_sorted.numel()) & (keys_sorted[pos] == rev_keys)
            alpha_rev = torch.zeros_like(alpha_forward)
            # gather α on matched reverse edges
            alpha_rev[match] = alpha_forward[order[pos[match]]]
            w_pos = torch.minimum(alpha_forward, alpha_rev)       # mutual weight
        else:
            w_pos = alpha_forward

        # Keep only edges with positive weight
        keep = w_pos > 0
        if not torch.any(keep):
            zero = svals.new_tensor(0.0, requires_grad=True)
            return zero, {"anchors_with_pos": 0, "loss": 0.0, "E_pos": 0, "E_neg": int(sim_neg.numel())}

        row_pos = row_pos[keep]
        sim_pos = sim_pos[keep]
        w_pos   = w_pos[keep]

        # --- Per-row NT-Xent: vectorized log-sum-exp with segmented reductions ---
        # Numerator per row i: log Σ_{j∈P(i)} w_ij * exp(sim_ij/τ)
        s_pos_tau = sim_pos / self.tau
        # stable: center by per-row max over positives
        max_pos_tau = _segment_amax(row_pos, s_pos_tau, num_segments=N, init=-float("inf"))
        exp_num = torch.exp(s_pos_tau - max_pos_tau[row_pos]) * torch.clamp(w_pos, min=self.eps)
        sum_num = _segment_sum(row_pos, exp_num, num_segments=N)
        log_num = torch.log(sum_num + self.eps) + max_pos_tau

        # Denominator per row i: log Σ_{q∈P(i)∪N(i)} exp(sim_iq/τ)
        s_neg_tau = sim_neg / self.tau if sim_neg.numel() else sim_neg
        # need max over union: max( max_pos_tau, max_neg_tau )
        if sim_neg.numel():
            max_neg_tau = _segment_amax(row_neg, s_neg_tau, num_segments=N, init=-float("inf"))
            denom_max = torch.maximum(max_pos_tau, max_neg_tau)
        else:
            denom_max = max_pos_tau

        # sum exp over positives (centered by denom_max)
        exp_pos_den = torch.exp(s_pos_tau - denom_max[row_pos])
        sum_pos_den = _segment_sum(row_pos, exp_pos_den, num_segments=N)

        # sum exp over negatives
        if sim_neg.numel():
            exp_neg_den = torch.exp(s_neg_tau - denom_max[row_neg])
            sum_neg_den = _segment_sum(row_neg, exp_neg_den, num_segments=N)
            sum_den = sum_pos_den + sum_neg_den
        else:
            sum_den = sum_pos_den
        log_den = torch.log(sum_den + self.eps) + denom_max

        # Consider only rows that have at least one positive
        count_pos = torch.bincount(row_pos, minlength=N)
        valid = count_pos > 0
        if not torch.any(valid):
            zero = svals.new_tensor(0.0, requires_grad=True)
            return zero, {"anchors_with_pos": 0, "loss": 0.0, "E_pos": int(row_pos.numel()), "E_neg": int(sim_neg.numel())}

        per_row = -(log_num[valid] - log_den[valid])
        loss = per_row.mean()

        info = {
            "loss": float(loss.detach().cpu()),
            "rows_with_pos": int(valid.sum().item()),
            "E_pos": int(row_pos.numel()),
            "E_neg": int(sim_neg.numel()),
            "temperature": self.tau,
            "tau_soft": self.tau_soft,
            "mutual": self.require_mutual,
        }
        return loss, info

In [ ]:
from __future__ import annotations
from dataclasses import dataclass
from typing import Dict, Optional, Tuple, Union, Literal
import torch
import torch.nn.functional as F

# --- SimpleCOO / SimpleCSR and segmented utils come from your snippet above ---


class PUAUCGraphLoss(torch.nn.Module):
    """
    NU-AUC (Negative–Unlabeled) pairwise ranking on a sparse similarity graph.

    RH 2025

    Summary
    -------
    - Inputs:
        sim    : sparse similarities (CSR/COO), values are the pairwise *scores* s_{ij}.
        s_sesh : same sparsity; boolean/binary (1 = same-session ⇒ labeled NEGATIVE).
      Cross-session edges are *unlabeled* (mixture of true pos/neg).

    - Objective (default squared-hinge AUC surrogate):
        For each row i with U(i)=cross-session edges and N(i)=same-session edges,
            L_i = sum_{u in U(i)} sum_{n in N(i)} [ m - (s_u - s_n) ]_+^2
        L = mean_i L_i over rows with |U(i)|>0 and |N(i)|>0.
      This directly maximizes the ranking gap s_u > s_n without encouraging endpoint spikes.

    - Computational trick (no dense pairwise expansion):
        Sort s_n per row; use searchsorted + suffix sums to evaluate all violating pairs in
        O((|U|+|N|) log |N|) per row:
            Let t = lower_bound(s_n >= s_u - m). For k>=t:
                (m + s_n - s_u)^2 = (m - s_u)^2 + 2(m - s_u)s_n + s_n^2
            Use suffix counts/sums/sumsq to aggregate in O(1) per u.

    - Optional 'logistic' surrogate (accurate but slower):
        L_i = sum_{u} sum_{n} softplus( (s_n - s_u) / tau_pair ).
        Use only when degrees are small, or cap per-row edges with max_* arguments.

    Why NU-AUC?
    -----------
    With labeled negatives and unlabeled cross-session edges, minimizing NU-AUC is
    theoretically equivalent (up to a positive affine transform) to minimizing PN-AUC,
    so no class-prior estimate is required.  (Sakai et al. 2017/2018; Xie et al. 2018)
    See Eq. (5)/(7) and the PU/NU–PN linear relation.  This neatly avoids the class-
    prior estimation step common in PU AUC optimization.   [oai_citation:7‡arXiv](https://arxiv.org/pdf/1705.01708?utm_source=chatgpt.com) [oai_citation:8‡AAAI](https://cdn.aaai.org/ojs/11812/11812-13-15340-1-2-20201228.pdf)

    Args:
        surrogate (Literal['sqhinge','logistic']):
            Pairwise loss. 'sqhinge' is fast and robust; 'logistic' is precise but O(|U||N|).
        margin (float):
            Margin m for squared hinge (default 0.0).
        tau_pair (float):
            Temperature for logistic pairwise loss (softplus((s_n-s_u)/tau_pair)).
        max_pos_per_row (Optional[int]):
            If set, keep at most this many cross-session edges per row (top by score).
        max_neg_per_row (Optional[int]):
            If set, keep at most this many same-session edges per row (top by score).
        assume_csr_sorted (bool):
            Assume CSR columns sorted within each row (recommended).
        eps (float):
            Numerical epsilon.

    Returns:
        loss (torch.Tensor):
            Scalar objective.
        info (Dict[str, float]):
            Diagnostics (counts, pairs evaluated, settings).
    """

    def __init__(
        self,
        *,
        surrogate: Literal["sqhinge", "logistic"] = "sqhinge",
        margin: float = 0.0,
        tau_pair: float = 0.05,
        max_pos_per_row: Optional[int] = None,
        max_neg_per_row: Optional[int] = None,
        assume_csr_sorted: bool = True,
        eps: float = 1e-12,
    ) -> None:
        super().__init__()
        self.surrogate = surrogate
        self.margin = float(margin)
        self.tau_pair = float(tau_pair)
        self.max_pos_per_row = int(max_pos_per_row) if max_pos_per_row is not None else None
        self.max_neg_per_row = int(max_neg_per_row) if max_neg_per_row is not None else None
        self.assume_csr_sorted = bool(assume_csr_sorted)
        self.eps = float(eps)

    @torch.no_grad()
    def _ensure_csr(self, A: Union[SimpleCSR, SimpleCOO]) -> SimpleCSR:
        return A if isinstance(A, SimpleCSR) else A.to_csr()

    def _row_slice(self, row_ptr: torch.Tensor, col_idx: torch.Tensor, vals: torch.Tensor, row: int) -> Tuple[torch.Tensor, torch.Tensor]:
        s, e = int(row_ptr[row].item()), int(row_ptr[row + 1].item())
        return col_idx[s:e], vals[s:e]

    def forward(
        self,
        sim: Union[SimpleCSR, SimpleCOO],
        s_sesh: Union[SimpleCSR, SimpleCOO],
    ) -> Tuple[torch.Tensor, Dict[str, float]]:
        sim_csr = self._ensure_csr(sim)
        ss_csr  = self._ensure_csr(s_sesh)
        if not (torch.equal(sim_csr.row_ptr, ss_csr.row_ptr) and torch.equal(sim_csr.col_idx, ss_csr.col_idx)):
            raise ValueError("sim and s_sesh must share identical CSR structure (row_ptr & col_idx).")

        row_ptr, col_idx, svals, ssv = sim_csr.row_ptr, sim_csr.col_idx, sim_csr.val, ss_csr.val
        N, M = sim_csr.shape
        assert N == M, "Expect square similarity matrix."

        # device = svals.device
        dtype  = svals.dtype

        # Build row ids for edges and masks
        deg = row_ptr[1:] - row_ptr[:-1]  # (N,)
        row_e = torch.repeat_interleave(torch.arange(N, device='cpu', dtype=torch.int64), deg)  # (nnz,)
        same_mask  = ssv.to(torch.bool).to('cpu')     # same-session (labeled negatives)
        cross_mask = ~same_mask             # cross-session (unlabeled)

        # Partition by type
        row_u = row_e[cross_mask]; s_u = svals[cross_mask].to(dtype)   # U edges and scores
        row_n = row_e[same_mask];  s_n = svals[same_mask].to(dtype)    # N edges and scores

        # Rows that have BOTH U and N
        has_u = torch.bincount(row_u, minlength=N) > 0
        has_n = torch.bincount(row_n, minlength=N) > 0
        valid_rows = torch.nonzero(has_u & has_n, as_tuple=False).flatten()

        if valid_rows.numel() == 0:
            zero = svals.new_tensor(0.0, requires_grad=True)
            return zero, {"rows_with_pairs": 0, "pairs_evaluated": 0, "loss": 0.0}

        total_pairs = 0
        total_rows  = 0
        losses: list[torch.Tensor] = []

        # Per-row fast evaluation (degrees are variable; this avoids dense pairwise expansion)
        for i in valid_rows.tolist():
            # Slices
            cols_i, sims_i = self._row_slice(row_ptr, col_idx, svals, i)
            # mask them by type
            mask_u = (~ssv[row_ptr[i]:row_ptr[i+1]].to(torch.bool))
            mask_n = (~mask_u)

            s_u_i = sims_i[mask_u]
            s_n_i = sims_i[mask_n]

            # Cap per-row work if requested
            if self.max_pos_per_row is not None and s_u_i.numel() > self.max_pos_per_row:
                # keep top similarities (hard negatives/strong positives for ranking)
                v, idx = torch.topk(s_u_i, k=self.max_pos_per_row, largest=True, sorted=False)
                s_u_i = v
            if self.max_neg_per_row is not None and s_n_i.numel() > self.max_neg_per_row:
                v, idx = torch.topk(s_n_i, k=self.max_neg_per_row, largest=True, sorted=False)
                s_n_i = v

            if s_u_i.numel() == 0 or s_n_i.numel() == 0:
                continue

            total_rows += 1

            if self.surrogate == "sqhinge":
                # Sort negatives ascending; precompute suffix stats
                s_n_sorted, _ = torch.sort(s_n_i)  # (M,)
                M = s_n_sorted.numel()
                # Suffix counts, sums, sumsq
                ones = torch.ones_like(s_n_sorted)
                suff_cnt = torch.cumsum(ones.flip(0), dim=0).flip(0)                   # (#violations)
                suff_sum = torch.cumsum(s_n_sorted.flip(0), dim=0).flip(0)            # sum s_n over tail
                suff_s2  = torch.cumsum((s_n_sorted**2).flip(0), dim=0).flip(0)       # sum s_n^2 over tail

                # For each u, find first violating index t where s_n >= s_u - m
                thresh = s_u_i - self.margin
                # lower_bound on sorted s_n
                t = torch.searchsorted(s_n_sorted, thresh.clamp_min(-1e6))            # (K,)
                # mask where any violators exist
                has_viols = t < M
                if has_viols.any():
                    t_idx = t[has_viols]
                    # gather suffix stats at t
                    cnt = suff_cnt[t_idx]
                    sum_sn = suff_sum[t_idx]
                    sum_s2 = suff_s2[t_idx]
                    # constant term per u: (m - s_u)^2 * cnt
                    d = (self.margin - s_u_i[has_viols])
                    term_const = (d * d) * cnt
                    term_lin   = 2.0 * d * sum_sn
                    term_quad  = sum_s2
                    loss_i = (term_const + term_lin + term_quad).sum()
                else:
                    loss_i = s_u_i.new_tensor(0.0)

                losses.append(loss_i / (s_u_i.numel() * s_n_sorted.numel()))

                total_pairs += int(s_u_i.numel()) * int(s_n_sorted.numel())

            else:  # 'logistic' (accurate but O(|U||N|))
                # To avoid allocating a huge matrix, chunk U if necessary
                K, M = int(s_u_i.numel()), int(s_n_i.numel())
                total_pairs += K * M
                # heuristic chunking for safety
                max_block = 65536  # ~64k pairwise ops per block
                block = max(1, max_block // max(1, M))
                loss_i = s_u_i.new_tensor(0.0)
                for start in range(0, K, block):
                    end = min(start + block, K)
                    su = s_u_i[start:end].unsqueeze(1)            # (b,1)
                    sn = s_n_i.unsqueeze(0)                        # (1,M)
                    loss_i = loss_i + F.softplus((sn - su) / max(self.tau_pair, 1e-6)).sum()
                losses.append(loss_i / (K * M))

        if not losses:
            zero = svals.new_tensor(0.0, requires_grad=True)
            return zero, {"rows_with_pairs": 0, "pairs_evaluated": 0, "loss": 0.0}

        loss = torch.stack(losses).mean()

        info = {
            "loss": float(loss.detach().cpu()),
            "rows_with_pairs": total_rows,
            "pairs_evaluated": int(total_pairs),
            "surrogate": self.surrogate,
            "margin": self.margin,
            "tau_pair": self.tau_pair,
            "max_pos_per_row": self.max_pos_per_row or 0,
            "max_neg_per_row": self.max_neg_per_row or 0,
        }
        return loss, info

In [9]:
import torch

In [11]:
import numpy as np
import scipy.sparse
import torch
import torch.nn.functional as F

class VectorizedPUAUCLoss(torch.nn.Module):
    """
    Method 3: Vectorized GPU Loss for PUAUC (Positive-Unlabeled AUC).
    
    This method assumes:
      1. 'Same-session' edges (ssit) are hard negatives (cannot be the same neuron).
      2. 'Cross-session' edges (adj & ~ssit) are unlabeled candidates (potential positives).
      
    It optimizes the objective: Sim(Cross) > Sim(Same) + Margin.
    """
    
    def __init__(
        self,
        adj: scipy.sparse.csr_matrix,
        ssit: scipy.sparse.csr_matrix,
        device: torch.device,
        margin: float = 0.1,
        K: int = 2,          # Number of cross-session (unlabeled) edges to sample per row
        M: int = 8,          # Number of same-session (negative) edges to sample per row
        R_sample: int = 8000, # Number of rows to sample per resampling step
        resample_every: int = 25,
        seed: int = 0
    ):
        super().__init__()
        self.margin = margin
        self.K = K
        self.M = M
        self.R_sample = R_sample
        self.resample_every = resample_every
        self.device = device
        self.seed = seed
        self.step_counter = 0
        
        # --- 1. Graph Preprocessing (CPU) ---
        # Ensure identical structure
        if not (adj.indptr.shape == ssit.indptr.shape and 
                np.array_equal(adj.indptr, ssit.indptr) and 
                np.array_equal(adj.indices, ssit.indices)):
            raise ValueError("adj and ssit must share identical CSR structure (indptr/indices).")

        self.N = adj.shape[0]
        self.indptr = adj.indptr
        
        # Create canonical edge lists
        # edge_row: source node index for every edge
        self.edge_row_cpu = np.repeat(np.arange(self.N, dtype=np.int64), np.diff(self.indptr))
        self.edge_col_cpu = adj.indices.astype(np.int64)
        
        # Boolean mask: True if edge is same-session, False if cross-session
        self.same_mask_cpu = ssit.data.astype(bool)
        
        # --- 2. Filter Valid Rows ---
        # We need rows that have at least K cross edges and M same edges
        deg = np.diff(self.indptr)
        deg_same = np.add.reduceat(self.same_mask_cpu, self.indptr[:-1])
        deg_cross = deg - deg_same
        
        self.valid_rows = np.where((deg_cross >= K) & (deg_same >= M))[0]
        if len(self.valid_rows) == 0:
            raise RuntimeError(f"No valid rows found with K={K} (cross) and M={M} (same). Try reducing M.")
            
        print(f"VectorizedPUAUCLoss initialized: {len(self.valid_rows)}/{self.N} valid rows. "
              f"Sampling {R_sample} rows every {resample_every} steps.")

        # --- 3. Move Static Graph Data to GPU ---
        # We compute cosine similarity for ALL edges on the GPU, so we need these tensors resident.
        self.edge_row_t = torch.as_tensor(self.edge_row_cpu, dtype=torch.int64, device=device)
        self.edge_col_t = torch.as_tensor(self.edge_col_cpu, dtype=torch.int64, device=device)
        
        # Placeholders for active indices (will be set by resample)
        self.U_idx_t = None # Indices into edge list for Unlabeled (cross) edges
        self.N_idx_t = None # Indices into edge list for Negative (same) edges
        
        # Initial sampling
        self.resample()

    def resample(self):
        """
        Samples a fixed set of R rows, and for each row samples K cross edges and M same edges.
        This runs on CPU using numpy for efficiency, then transfers indices to GPU.
        """
        rng = np.random.default_rng(self.seed + self.step_counter)
        
        # Select rows
        if self.R_sample >= len(self.valid_rows):
            rows_sel = self.valid_rows
        else:
            rows_sel = rng.choice(self.valid_rows, size=self.R_sample, replace=False)
            
        # Containers for edge indices
        # We want to store the INDEX into the edge array (0..E-1), not the column node ID
        U_idx = np.empty((len(rows_sel), self.K), dtype=np.int64)
        N_idx = np.empty((len(rows_sel), self.M), dtype=np.int64)
        
        # Loop over sampled rows to pick specific edges
        # Note: This Python loop is acceptable because R_sample is relatively small (e.g. 8000)
        # and it only runs once every 'resample_every' iterations.
        for t, r in enumerate(rows_sel):
            s, e = self.indptr[r], self.indptr[r+1]
            
            # These are indices into the edge list (0..E)
            edges_range = np.arange(s, e, dtype=np.int64)
            sm = self.same_mask_cpu[s:e]
            
            same_edges = edges_range[sm]
            cross_edges = edges_range[~sm]
            
            # Random choice
            U_idx[t] = rng.choice(cross_edges, size=self.K, replace=False)
            N_idx[t] = rng.choice(same_edges, size=self.M, replace=False)
            
        # Transfer to GPU
        self.U_idx_t = torch.as_tensor(U_idx, dtype=torch.int64, device=self.device)
        self.N_idx_t = torch.as_tensor(N_idx, dtype=torch.int64, device=self.device)

    def forward(self, embeddings: torch.Tensor) -> torch.Tensor:
        """
        Args:
            embeddings: (N, D) tensor of ALL node embeddings.
        Returns:
            Scalar loss.
        """
        # 1. Periodic Resampling
        if self.step_counter > 0 and (self.step_counter % self.resample_every) == 0:
            self.resample()
        self.step_counter += 1
        
        # 2. Compute similarity on ALL edges (Vectorized GPU operation)
        #    This is fast: O(E * D)
        #    zi = embeddings[row], zj = embeddings[col]
        #    sim = (zi * zj).sum(1)
        #    Using F.normalize ensures cosine similarity.
        emb_norm = F.normalize(embeddings, dim=1, eps=1e-12)
        zi = emb_norm[self.edge_row_t]
        zj = emb_norm[self.edge_col_t]
        sim_vals = (zi * zj).sum(dim=1) # Shape: (num_edges,)
        
        # 3. Gather pre-sampled similarities
        #    U_idx_t shape: (R, K) -> s_u shape: (R, K)
        #    N_idx_t shape: (R, M) -> s_n shape: (R, M)
        s_u = sim_vals[self.U_idx_t] 
        s_n = sim_vals[self.N_idx_t]
        
        # 4. Vectorized Squared Hinge Loss
        #    Objective: s_u > s_n + margin
        #    Violation: d = (s_n + margin) - s_u
        #    We minimize sum(max(0, d)^2)
        
        # Broadcast subtraction: (R, 1, M) - (R, K, 1) -> (R, K, M)
        d = self.margin + s_n.unsqueeze(1) - s_u.unsqueeze(2)
        z = torch.clamp(d, min=0.0)
        loss = (z * z).mean()
        
        return loss

In [12]:
bc = scipy.sparse.vstack(b)
bc2 = bc[:].copy().astype(np.bool_)
adj = (bc2 @ bc2.T).tocsr()
adj[range(adj.shape[0]), range(adj.shape[1])] = False  # remove self-loops
adj.eliminate_zeros()

/Users/richardhakim/Documents/virtual_environments/roicat/lib/python3.11/site-packages/scipy/sparse/_index.py:210: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil and dok are more efficient.
  self._set_arrayXarray(i, j, x)


In [13]:
sbis = roicat.helpers.masked_pairwise_similarity_dense(
    X=torch.as_tensor(sb),
    adjacency=adj[:][:, :],
    metric='dot',
    device='cpu',
    accum_dtype=torch.float32,
    return_tensor=False,
    indices_device='cpu',
    edges_per_chunk=1024,
    eps=1e-12,
)
s_sesh_inv = scipy.sparse.coo_matrix((sbis[2].detach().cpu().numpy(), (sbis[0].detach().cpu().numpy(), sbis[1].detach().cpu().numpy())), shape=sbis[3]).tocsr()
s_sesh_inv.sort_indices()
# s_sesh_inv.eliminate_zeros()

# ssit = roicat.helpers.scipy_sparse_to_torch_coo(s_sesh_inv).coalesce()
ssit = s_sesh_inv.copy()

In [ ]:
# ws = Wasserstein1SeparationLoss(
#     lambda_diff_to_one=0.0,
#     lambda_bimodal_all=0.00,
#     clamp_inputs=True,
# )

In [ ]:
# scl = SparseContrastiveLoss(
#     temperature=0.2,
#     tau_soft=0.02,
#     preselect_k=64,
#     max_negs_per_anchor=None,
#     assume_row_sorted=False,
#     mutual=False,
#     eps=1e-12,
# )

# scl = MutualSoftmaxNTXentSparse(
#     temperature=0.2,
#     tau_soft=0.02,
#     require_mutual=True,
#     eps=1e-12,
# )

puauc = PUAUCGraphLoss(
    surrogate="logistic",
    margin=0.1,
    tau_pair=0.05,
    max_pos_per_row=2,
    max_neg_per_row=256,
    assume_csr_sorted=True,
    eps=1e-12,
)


In [ ]:
def pipeline(rl, adj, ssit, device):
    # ssit = roicat.helpers.scipy_sparse_to_torch_coo(ssit)
    # ssit = ssit.tocoo()
    
    rlst = roicat.helpers.masked_pairwise_similarity_dense(
        X=rl[:],
        adjacency=adj[:][:, :],
        metric='cosine',
        device=device,
        accum_dtype=torch.float32,
        return_tensor=False,
        indices_device=device,
        edges_per_chunk=102400,
        eps=1e-12,
    # ).coalesce()
    )
    # idx = multiindex_argsort(
    #     data=torch.stack(rlst[:2], dim=0),
    #     dim=1,
    # )
    # v = rlst[2][idx]
    # v_all = 1 - ((v + 1)/1)
    # v_diff = v_all[ssit.data]
    # v_diff = 1 - (((v[ssit.data] + 1) / 2))
    # assert np.allclose(rlst[1], ssit.row) and np.allclose(rlst[0], ssit.col), "Mismatch in adjacency structure"

    # # v_all = 1 - ((rlst.values() + 1)/2)
    # # v_diff = 1 - (((rlst * ssit).coalesce().values() + 1) / 2)
    # v_all = 1 - ((rlst[2] + 1)/2)
    # v_diff = 1 - (((rlst[2][ssit.data]) + 1) / 2)

    # v_all = -rlst.values()
    # v_diff = -(rlst * ssit).coalesce().values()

    # return -torch.nn.functional.kl_div(
    #     input=v_all[np.random.randint(0, len(v_all), size=v_diff.shape[0])],
    #     target=v_diff,
    #     reduction='mean',
    # )
    # out = ws.forward(
    #     v_all=v_all,
    #     v_diff=v_diff,
    #     grid_size=1024,
    # )
    
    # rlst_c = coo_to_csr(
    #     row=rlst[0],
    #     col=rlst[1],
    #     values=rlst[2],
    #     n_rows=rlst[3][0],
    #     n_cols=rlst[3][1],
    #     coalesce=False,
    # )
    # ssit_c = coo_to_csr(
    #     row=ssit.row,
    #     col=ssit.col,
    #     values=ssit.val,
    #     n_rows=ssit.shape[0],
    #     n_cols=ssit.shape[1],
    #     coalesce=False,
    # )
    
    rlst_c = SimpleCOO(
        row=rlst[0],
        col=rlst[1],
        # val=torch.nn.functional.gelu(rlst[2]),
        val=rlst[2],
        shape=rlst[3],
    )
    rlst_c.sort_indices_()
    rlst_c = rlst_c.to_csr()
    rlst_c.val, rlst_c.row_ptr, rlst_c.col_idx = rlst_c.val.to(device), rlst_c.row_ptr.to('cpu'), rlst_c.col_idx.to('cpu')

    # out = scl.forward(
    #     row_ptr=rlst_c[0],
    #     col_idx=rlst_c[1],
    #     sim_vals=rlst_c[2],
    #     same_sesh_vals=ssit[2],
    #     anchors=None,
    #     embeddings=None,
    # )
    # out = scl.forward(
    #     sim=rlst_c,
    #     s_sesh=ssit,
    # )
    out = puauc.forward(
        sim=rlst_c,
        s_sesh=ssit,
    )
    return out[0]
    # return v_all.mean()

In [ ]:
device = 'cpu'
rl2 = torch.as_tensor(rl.detach().cpu(), dtype=torch.float32, device=device)
rl2.requires_grad = False

n_layers = 1
lambda_weight_decay = 1e-4
lr = 1e-2
batch_size = 50000

ls = {f'linear_{ii}': torch.nn.Linear(rl2.shape[1]//(4**(ii)), rl2.shape[1]//(4**(ii+1)), bias=False, device=device, dtype=torch.float32) for ii in range(n_layers)}
## initialize with xavier
for layer in ls.values():
    torch.nn.init.xavier_uniform_(layer.weight)
act = torch.nn.GELU()
optim = torch.optim.Adam((param for layer in ls.values() for param in layer.parameters()), lr=lr, weight_decay=lambda_weight_decay)
def forward(x):
    for layer in ls.values():
        # x = act(layer(x))
        x = layer(x)
    return x

n_samples = rl2.shape[0]

# idx_batch = np.random.choice(n_samples, batch_size, replace=False)

# ssit2 = roicat.helpers.scipy_sparse_to_torch_coo(ssit).coalesce().to(device)

logger = []
for ii in range(10000):
    idx_batch = np.random.choice(n_samples, batch_size, replace=False) if batch_size < n_samples else np.arange(n_samples)
    # idx_batch = np.arange(n_samples)
    
    # ssit_coo = ssit[idx_batch][:, idx_batch].tocoo()
    # ssit2 = coo_to_csr(
    #     row=torch.as_tensor(ssit_coo.row),
    #     col=torch.as_tensor(ssit_coo.col),
    #     values=torch.as_tensor(ssit_coo.data),
    #     n_rows=ssit_coo.shape[0],
    #     n_cols=ssit_coo.shape[1],
    #     coalesce=False,
    # )
    # ssit2 = ssit
    ssit2 = ssit[idx_batch][:, idx_batch]
    ssit2.sort_indices()
    ssit2 = SimpleCSR.from_scipy(ssit2)
    ssit2.val, ssit2.row_ptr, ssit2.col_idx = ssit2.val.to(device), ssit2.row_ptr.to('cpu'), ssit2.col_idx.to('cpu')
    
    # emb = forward(rl2)
    emb = forward(rl2)[idx_batch]
    
    # loss = pipeline(emb, adj, ssit2, device)
    loss = pipeline(emb, adj[idx_batch][:, idx_batch], ssit2, device)
    
    
    loss.backward()
    optim.step()
    optim.zero_grad()

    logger.append({'loss': loss.item()})
    print(f"iter: {ii}, loss: {logger[-1]['loss']:.8f}, ")

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
plt.figure()
plt.plot([log['loss'] for log in logger], label='Loss')
plt.xlabel('Iteration')
plt.ylabel('Loss')
plt.title('Training Loss Over Time')
plt.legend()
plt.show()

In [ ]:
emb = forward(rl2.detach())

rlst = roicat.helpers.masked_pairwise_similarity_dense(
    X=emb[:],
    adjacency=adj[:][:, :],
    metric='cosine',
    device=device,
    accum_dtype=torch.float32,
    return_tensor=True,
    indices_device=device,
    edges_per_chunk=1024,
    eps=1e-12,
).coalesce()

sbis = roicat.helpers.masked_pairwise_similarity_dense(
    X=torch.as_tensor(sb),
    adjacency=adj[:][:, :],
    metric='dot',
    device='cpu',
    accum_dtype=torch.float32,
    return_tensor=False,
    indices_device='cpu',
    edges_per_chunk=1024,
    eps=1e-12,
)
# sbis
s_sesh_inv = scipy.sparse.coo_matrix((sbis[2].detach().cpu().numpy(), (sbis[0].detach().cpu().numpy(), sbis[1].detach().cpu().numpy())), shape=sbis[3]).tocsr()
s_sesh_inv.eliminate_zeros()
ssit2 = roicat.helpers.scipy_sparse_to_torch_coo(s_sesh_inv).coalesce()


# v_all = 1 - ((rlst.values() + 1)/2)
# v_diff = 1 - (((rlst * ssit2).coalesce().values() + 1) / 2)
v_all = rlst.values()
v_diff = (rlst * ssit2).coalesce().values() 
import matplotlib.pyplot as plt

In [ ]:
plt.figure()
plt.hist(v_all.detach().cpu().numpy()**1, bins=300, label='All', alpha=0.5, density=True);
plt.hist(v_diff.detach().cpu().numpy()**1, bins=300, label='Different', alpha=0.5, density=True);
plt.legend();
# plt.ylim(0, 1e1)
# plt.yscale('log')

In [ ]:
plt.figure()
plt.hist(v_all.detach().cpu().numpy()**1, bins=300, label='All', alpha=0.5, density=True);
plt.hist(v_diff.detach().cpu().numpy()**1, bins=300, label='Different', alpha=0.5, density=True);
plt.legend();
# plt.ylim(0, 1e1)
# plt.yscale('log')

In [ ]:
emb = forward(rl2.detach())

rls = roicat.helpers.masked_pairwise_similarity_dense(
    X=emb[:],
    adjacency=adj[:][:, :],
    metric='cosine',
    device=device,
    accum_dtype=torch.float32,
    return_tensor=False,
    indices_device=device,
    edges_per_chunk=1024,
    eps=1e-12,
)
    
s_NN = scipy.sparse.coo_matrix((rls[2].detach().cpu().numpy(), (rls[0].detach().cpu().numpy(), rls[1].detach().cpu().numpy())), shape=rls[3]).tocsr()
d_conj = s_NN.copy()
# d_conj.data = 1 - ((d_conj.data + 1)/2)
d_conj.data = d_conj.data

dsl = roicat.tracking.clustering.DistributionSeparationLoss(
    num_bins=64,
    kernel="triangular",
    temperature=None,
    chunk_size=None,
    scale_diff_by_ratio=True,
    sigma_bins_smooth=None,
    overlap_weight=1.0,
    margin_weight=0.5,
    mass_weight=0.0,
    margin=0.20,
    mass_target=0.05,
    device=None,
    dtype=torch.float32,
)

In [ ]:
out = dsl.forward(
    d_conj=d_conj.power(1),
    s_sesh_inv=s_sesh_inv,
)
out[1].keys()


plt.figure()
plt.plot(out[1]['centers'], out[1]['dens_all'])
plt.plot(out[1]['centers'], out[1]['dens_diff'])
plt.plot(out[1]['centers'], out[1]['dens_same']*15)
plt.ylim(0, None)

In [ ]:
s_NN

In [ ]:
test = SimpleCOO(row=rls[0], col=rls[1], val=rls[2].detach(), shape=rls[3]).to_scipy().tocsr()

In [ ]:
test

In [ ]:
np.allclose(s_NN.data, test.data)

In [ ]:
test2 = test.copy()

In [ ]:
test2.data[test2.data < 0] = 0.0

In [ ]:
s_NN2 = s_NN.copy()

In [ ]:
s_NN2.eliminate_zeros()
test2.eliminate_zeros()

In [ ]:
s_NN2, test2

In [ ]:
plt.figure()
plt.hist(s_NN2.data, bins=100, density=True, alpha=0.5, label='s_NN data');
# plt.hist(test.data, bins=100, density=True, alpha=0.5, label='test data');

In [ ]:
s_NN_d = s_NN[:5000][:, :5000].toarray()
test_d = test[:5000][:, :5000].toarray()

In [ ]:
np.allclose(s_NN.data, test.data)

In [21]:
# %load_ext autoreload
# %autoreload 2
import roicat
import roicat.helpers
import roicat.tracking.clustering

from pathlib import Path

import scipy.sparse
import torch
import numpy as np

In [ ]:
# ## temporarily save feature vectors:
# dir_save = "/Users/richardhakim/Desktop/ROICaT_temp"
# Path(dir_save).mkdir(parents=True, exist_ok=True)

# roicat.helpers.pickle_save(obj=aligner.ROIs_aligned, filepath=str(Path(dir_save) / 'ROIs_aligned.pkl'))
# roicat.helpers.pickle_save(obj=blurrer.ROIs_blurred, filepath=str(Path(dir_save) / 'ROIs_blurred.pkl'))
# roicat.helpers.pickle_save(obj=roinet.latents, filepath=str(Path(dir_save) / 'roinet_latents.pkl'))
# roicat.helpers.pickle_save(obj=swt.latents, filepath=str(Path(dir_save) / 'swt_latents.pkl'))
# roicat.helpers.pickle_save(obj=data.session_bool, filepath=str(Path(dir_save) / 'session_bool.pkl'))

In [ ]:
## temporarily save feature vectors:
dir_save = "/Users/richardhakim/Desktop/ROICaT_temp"
Path(dir_save).mkdir(parents=True, exist_ok=True)

r = roicat.helpers.pickle_load(filepath=str(Path(dir_save) / 'ROIs_aligned.pkl'))
b = roicat.helpers.pickle_load(filepath=str(Path(dir_save) / 'ROIs_blurred.pkl'))
rl = roicat.helpers.pickle_load(filepath=str(Path(dir_save) / 'roinet_latents.pkl'))
sl = roicat.helpers.pickle_load(filepath=str(Path(dir_save) / 'swt_latents.pkl'))
sb = roicat.helpers.pickle_load(filepath=str(Path(dir_save) / 'session_bool.pkl'))

In [ ]:
# bc = scipy.sparse.vstack(b)
# bc2 = bc[:].copy().astype(np.bool_)
# adj = bc2 @ bc2.T
# # bc3 = roicat.helpers.scipy_sparse_to_torch_coo(sp_array=bc[:], dtype=torch.float32).to('cpu')

In [23]:
bc = scipy.sparse.vstack(b)
bc2 = bc[:].copy().astype(np.bool_)
adj = (bc2 @ bc2.T).tocsr()
adj[range(adj.shape[0]), range(adj.shape[1])] = False  # remove self-loops
adj.eliminate_zeros()

/Users/richardhakim/Documents/virtual_environments/roicat/lib/python3.11/site-packages/scipy/sparse/_index.py:210: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil and dok are more efficient.
  self._set_arrayXarray(i, j, x)


In [24]:
rlst = roicat.helpers.masked_pairwise_similarity_dense(
    X=rl[:],
    adjacency=adj[:][:, :],
    metric='cosine',
    device='cpu',
    accum_dtype=torch.float32,
    return_tensor=True,
    indices_device='cpu',
    edges_per_chunk=1024,
    eps=1e-12,
).coalesce()

In [31]:
sbis = roicat.helpers.masked_pairwise_similarity_dense(
    X=torch.as_tensor(sb),
    adjacency=adj[:][:, :],
    metric='dot',
    device='cpu',
    accum_dtype=torch.float32,
    return_tensor=False,
    indices_device='cpu',
    edges_per_chunk=1024,
    eps=1e-12,
)
s_sesh_inv = scipy.sparse.coo_matrix((sbis[2].detach().cpu().numpy(), (sbis[0].detach().cpu().numpy(), sbis[1].detach().cpu().numpy())), shape=sbis[3]).tocsr()
s_sesh_inv.sort_indices()
# s_sesh_inv.eliminate_zeros()

# ssit = roicat.helpers.scipy_sparse_to_torch_coo(s_sesh_inv).coalesce()
ssit = s_sesh_inv.copy()

In [26]:
v_all = 1 - ((rlst.values() + 1)/2)
v_diff = 1 - (((rlst * ssit).coalesce().values() + 1) / 2)

In [27]:
data_loaded = data.__dict__.copy()

In [28]:
# ================================
# Fast-load metadata needed for normalization
# ================================

import numpy as np

FOV_height = data_loaded.get('frame_height', data_loaded.get('FOV_height', 512))
FOV_width = data_loaded.get('frame_width', data_loaded.get('FOV_width', 1024))

n_sessions = data_loaded.get('n_sessions', sb.shape[1] if hasattr(sb, 'shape') else None)
centers_of_mass = data_loaded.get('centroids', None)
if centers_of_mass is None:
    centers_of_mass = data_loaded.get('centers_of_mass', None)

if centers_of_mass is None:
    def _centroids_from_csr(sf_csr, height, width):
        indptr = sf_csr.indptr
        indices = sf_csr.indices
        data = sf_csr.data
        n_roi = sf_csr.shape[0]
        y_cent = np.zeros(n_roi, dtype=np.float64)
        x_cent = np.zeros(n_roi, dtype=np.float64)
        for i in range(n_roi):
            s, e = indptr[i], indptr[i + 1]
            if s == e:
                continue
            idx = indices[s:e]
            w = data[s:e]
            y = idx // width
            x = idx % width
            tot = w.sum()
            if tot > 0:
                y_cent[i] = (y * w).sum() / tot
                x_cent[i] = (x * w).sum() / tot
        return np.stack([np.round(y_cent), np.round(x_cent)], axis=1).astype(np.int64)

    centers_of_mass = [_centroids_from_csr(sf, FOV_height, FOV_width) for sf in r]
    print("centers_of_mass missing in file; computed from ROIs_aligned")

print(f"FOV_height: {FOV_height}  FOV_width: {FOV_width}")
print(f"n_sessions: {n_sessions}")
print(f"centers_of_mass: {type(centers_of_mass)}")


FOV_height: 512  FOV_width: 1024
n_sessions: 35
centers_of_mass: <class 'list'>


In [32]:
# ================================
# Vectorized setup: CSR alignment + edges
# ================================

def csr_same_structure(A, B) -> bool:
    return (
        A.shape == B.shape
        and (A.indptr == B.indptr).all()
        and (A.indices == B.indices).all()
    )

if not csr_same_structure(adj, ssit):
    raise RuntimeError("Expected ssit to share identical CSR structure with adj. Vectorized pipeline assumes this.")

# Canonical edge list from CSR (adj / ssit share identical CSR structure)
N = adj.shape[0]
edge_row = np.repeat(np.arange(N, dtype=np.int64), np.diff(adj.indptr))
edge_col = adj.indices.astype(np.int64, copy=False)
edge_same_sesh = ssit.data.astype(bool, copy=False)

print(f"adj: N={N:,}  E={adj.nnz:,}")
print("edge_row:", edge_row.shape, "edge_col:", edge_col.shape)
print(f"edge_same_sesh: same={edge_same_sesh.sum():,} cross={(~edge_same_sesh).sum():,}")

# Rows with at least one same and one cross edge
_deg = np.diff(adj.indptr)
_deg_same = np.add.reduceat(edge_same_sesh, adj.indptr[:-1])
_deg_cross = _deg - _deg_same
valid_rows = np.where((_deg_same > 0) & (_deg_cross > 0))[0]
print(f"valid_rows: {len(valid_rows):,} / {N:,}")


adj: N=84,862  E=21,586,278
edge_row: (21586278,) edge_col: (21586278,)
edge_same_sesh: same=568,808 cross=21,017,470
valid_rows: 82,796 / 84,862


In [33]:
# ================================
# Vectorized helpers
# ================================

def sample_fixed_KM_indices(
    indptr: np.ndarray,
    same_mask: np.ndarray,
    valid_rows: np.ndarray,
    *,
    R: int,
    K: int,
    M: int,
    seed: int = 0,
):
    rng = np.random.default_rng(seed)

    deg = np.diff(indptr)
    deg_same = np.add.reduceat(same_mask, indptr[:-1])
    deg_cross = deg - deg_same

    eligible = valid_rows[(deg_cross[valid_rows] >= K) & (deg_same[valid_rows] >= M)]
    if len(eligible) == 0:
        raise RuntimeError(f"No eligible rows with K={K}, M={M}. Try smaller M.")

    rows_sel = eligible if R > len(eligible) else rng.choice(eligible, size=R, replace=False)

    U_idx = np.empty((len(rows_sel), K), dtype=np.int64)
    N_idx = np.empty((len(rows_sel), M), dtype=np.int64)

    for t, r in enumerate(rows_sel):
        s, e = indptr[r], indptr[r+1]
        edges = np.arange(s, e, dtype=np.int64)
        sm = same_mask[s:e]
        same_edges = edges[sm]
        cross_edges = edges[~sm]
        U_idx[t] = rng.choice(cross_edges, size=K, replace=False)
        N_idx[t] = rng.choice(same_edges, size=M, replace=False)

    return rows_sel, U_idx, N_idx


def sim_on_edges(emb: torch.Tensor, edge_row_t: torch.Tensor, edge_col_t: torch.Tensor) -> torch.Tensor:
    zi = torch.nn.functional.normalize(emb[edge_row_t], dim=1, eps=1e-12)
    zj = torch.nn.functional.normalize(emb[edge_col_t], dim=1, eps=1e-12)
    return (zi * zj).sum(dim=1)  # (E,)


def puauc_sqhinge_vectorized(sim_vals: torch.Tensor, U_idx: torch.Tensor, N_idx: torch.Tensor, margin: float) -> torch.Tensor:
    s_u = sim_vals[U_idx]  # (R, K)
    s_n = sim_vals[N_idx]  # (R, M)
    d = margin + s_n.unsqueeze(1) - s_u.unsqueeze(2)  # (R, K, M)
    z = torch.clamp(d, min=0.0)
    return (z * z).mean()

print("✓ vectorized helpers ready")


✓ vectorized helpers ready


In [37]:
# ================================
# Vectorized training harness
# ================================

import time

device = "cuda" if torch.cuda.is_available() else "cpu"
# device = "mps"
print("device:", device)

margin = 0.1
K = 2          # cross edges per row
M = 8          # same edges per row
R_train = 8000 # rows per iteration (tune)
R_eval  = 2000

resample_every = 25
eval_every = 10
seed0 = 123

rl2 = rl.detach().to(device).float()
proj = torch.nn.Linear(rl2.shape[1], rl2.shape[1]//4, bias=False, device=device)
torch.nn.init.xavier_uniform_(proj.weight)
optim = torch.optim.Adam(proj.parameters(), lr=1e-2, weight_decay=1e-4)

edge_row_t = torch.as_tensor(edge_row, dtype=torch.int64, device=device)
edge_col_t = torch.as_tensor(edge_col, dtype=torch.int64, device=device)

rows_eval, U_eval, N_eval = sample_fixed_KM_indices(
    indptr=adj.indptr,
    same_mask=edge_same_sesh,
    valid_rows=valid_rows,
    R=R_eval, K=K, M=M, seed=seed0 + 999,
)


def make_train_indices(seed: int):
    return sample_fixed_KM_indices(
        indptr=adj.indptr,
        same_mask=edge_same_sesh,
        valid_rows=valid_rows,
        R=R_train, K=K, M=M, seed=seed,
    )

rows_train, U_train, N_train = make_train_indices(seed0)

U_train_t = torch.as_tensor(U_train, dtype=torch.int64, device=device)
N_train_t = torch.as_tensor(N_train, dtype=torch.int64, device=device)
U_eval_t  = torch.as_tensor(U_eval,  dtype=torch.int64, device=device)
N_eval_t  = torch.as_tensor(N_eval,  dtype=torch.int64, device=device)

print(f"Train rows={len(rows_train):,}  Eval rows={len(rows_eval):,}  K={K}  M={M}")


def eval_loss():
    with torch.no_grad():
        emb = proj(rl2)
        sim_vals = sim_on_edges(emb, edge_row_t, edge_col_t)
        return float(puauc_sqhinge_vectorized(sim_vals, U_eval_t, N_eval_t, margin).detach().cpu())

n_iters = 100
train_losses = []
eval_losses = []
iter_times = []

for it in range(n_iters):
    it0 = time.time()

    if (it % resample_every) == 0 and it != 0:
        rows_train, U_train, N_train = make_train_indices(seed0 + it)
        U_train_t = torch.as_tensor(U_train, dtype=torch.int64, device=device)
        N_train_t = torch.as_tensor(N_train, dtype=torch.int64, device=device)

    emb = proj(rl2)
    sim_vals = sim_on_edges(emb, edge_row_t, edge_col_t)
    loss = puauc_sqhinge_vectorized(sim_vals, U_train_t, N_train_t, margin=margin)

    optim.zero_grad(set_to_none=True)
    loss.backward()
    optim.step()

    iter_times.append(time.time() - it0)
    train_losses.append(float(loss.detach().cpu()))

    if (it % eval_every) == 0 or it < 3:
        eval_losses.append((it, eval_loss()))
    
    print(f"iter {it:04d}  train_loss: {train_losses[-1]:.6f}  eval_loss: {eval_losses[-1][1]:.6f}  time: {iter_times[-1]:.3f}s")

print("✓ vectorized training complete")


device: cpu
Train rows=8,000  Eval rows=2,000  K=2  M=8


KeyboardInterrupt: 

In [ ]:
sim = roicat.tracking.similarity_graph.ROI_graph(
    n_workers=-1,  ## Number of CPU cores to use. -1 for all.
    # frame_height=data.FOV_height,
    frame_height=512,
    # frame_width=data.FOV_width,
    frame_width=1024,
    block_height=128,  ## size of a block
    block_width=128,  ## size of a block
    algorithm_nearestNeigbors_spatialFootprints='brute',  ## algorithm used to find the pairwise similarity for s_sf. ('brute' is slow but exact. See docs for others.)
    verbose=True,  ## Whether to print outputs
)

sim.visualize_blocks()

s_sf, s_NN, s_SWT, s_sesh = sim.compute_similarity_blockwise(
    # spatialFootprints=blurrer.ROIs_blurred,  ## Mask spatial footprints
    spatialFootprints=b,
    # features_NN=roinet.latents,  ## ROInet output latents
    features_NN=forward(rl).detach(),
    # features_SWT=swt.latents,  ## Scattering wavelet transform output latents
    features_SWT=sl,
    # ROI_session_bool=data.session_bool,  ## Boolean array of which ROIs belong to which sessions
    ROI_session_bool=sb,
    spatialFootprint_maskPower=1.0,  ##  An exponent to raise the spatial footprints to to care more or less about bright pixels
);

It is useful to normalize the similarity matrices using the local ROIs.

In [ ]:
s_sf, s_NN, s_SWT = sim.s_sf, sim.s_NN, sim.s_SWT

In [ ]:
sim.make_normalized_similarities(
    centers_of_mass=data.centroids,  ## ROI centroid positions
    features_NN=roinet.latents,  ## ROInet latents
    features_SWT=swt.latents,  ## SWT latents
    k_max=data.n_sessions*100,  ## Maximum number of nearest neighbors to consider for the normalizing distribution
    k_min=data.n_sessions*10,  ## Minimum number of nearest neighbors to consider for the normalizing distribution
    algo_NN='kd_tree',  ## Nearest neighbors algorithm to use
    device=DEVICE,
)

# Clustering

This step does the following:
1. Mix the similarity matrices into a single distance matrix
2. Prune the distance matrix to remove low probability connections
3. Perform clustering
4. Compute quality metrics

#### 1. Mix the similarity matrices into a single distance matrix

This step can be done either automatically, using the `clusterer.find_optimal_parameters_for_pruning` method, or manually by specifying the `kwargs_makeConjunctiveDistanceMatrix` dictionary. If you have a smaller total number of ROIs (<100 ROIs per session and/or <8 sessions), then it may be a good idea to manually play with the parameters in the next cell instead of using the automatic method.

<br></br>

##### Option A: Automatic Method
This step finds the optimal parameters to mix the similarity matrices by tuning mixing parameters to maximally separate the distributions of pairwise similarities for ROI pairs known to be different and ROI pairs that are likely matched.

Some of the details of underlying algorithm:
1. For each step in the optimization process, all the similarity matrices (`s_sf`, `s_NN_z`, `s_SWT_z`, `s_sesh`) are each passed through a sigmoid activation function that is parameterized (e.g. `'power_SF'`, `'sig_SF_kwargs'`), then all the similarity matrices are combined using a p-norm, where the 'p' is parameterized with `'p_norm'`. This results in a single conjunctive similarity matrix called `sConj` bounded between 0-1. 
2. `sConj` is converted into a distance matrix `dConj` (bounded from 1-0).
3. `dConj` is then passed through the objective function: The full distance matrix is separated into a few components. First, all pairs of ROIs that are known to be from 'different' sources because they are from the same session are separated out into a distribution of pairwise distances (`d_diff`). Second, we define pairs of ROIs that are likely to be from the same source as `d_same` = `d_all` - `d_diff`. The objective function is then the overlap between the `d_diff` and `d_same` distributions. 
4. The objective function is minimized by tuning the mixing parameters in `kwargs_makeConjunctiveDistanceMatrix`.
5. The output of this step is the optimal `kwargs_makeConjunctiveDistanceMatrix` dictionary.

<br></br>

##### Option B: Manual Method
You can also simply specify the `kwargs_makeConjunctiveDistanceMatrix` dictionary manually. This is useful if you have a good idea of what the optimal parameters are or if the automatic method is not working well. Uncomment the code block below to overwrite the `kwargs_mcdm_tmp` variable.

<br></br>

#### TROUBLESHOOTING FOR THIS STEP
- If you have any issues, just email Rich Hakim or open an issue on the github issues page.
- If you see: `'No crossover found, not plotting'`: Your data may not be easily separable. For some people, this is because the number of matching ROIs is very low compared to the number of non-matching ROIs. I recommend trying out the manual method.

In [ ]:
## Initialize the clusterer object by passing the similarity matrices in
clusterer = roicat.tracking.clustering.Clusterer(
    s_sf=sim.s_sf,
    s_NN_z=sim.s_NN_z,
    s_SWT_z=sim.s_SWT_z,
    s_sesh=sim.s_sesh,
    verbose=1,
)

In [ ]:
## Initialize the clusterer object by passing the similarity matrices in
clusterer = roicat.tracking.clustering.Clusterer(
    s_sf=sim.s_sf,
    s_NN_z=sim.s_NN,
    s_SWT_z=sim.s_SWT,
    s_sesh=sim.s_sesh,
    verbose=1,
)

#### Automatic method

In [ ]:
# Uncomment below to automatically find mixing parameters
kwargs_makeConjunctiveDistanceMatrix_best = clusterer.find_optimal_parameters_for_pruning(
    n_bins=None,  ## Number of bins to use for the histograms of the distributions. If None, then a heuristic is used.
    smoothing_window_bins=None,  ## Number of bins to use to smooth the distributions. If None, then a heuristic is used.
    kwargs_findParameters={
        'n_patience': 300,  ## Number of optimization epoch to wait for tol_frac to converge
        'tol_frac': 0.001,  ## Fractional change below which optimization will conclude
        'max_trials': 1200,  ## Max number of optimization epochs
        'max_duration': 60*10,  ## Max amount of time (in seconds) to allow optimization to proceed for
        'value_stop': 0.0,  ## Goal value. If value equals or goes below value_stop, optimization is stopped.
    },
    bounds_findParameters={
        'power_NN': [0.0, 2.],  ## Bounds for the exponent applied to s_NN
        'power_SWT': [0.0, 2.],  ## Bounds for the exponent applied to s_SWT
        'p_norm': [-5, -0.1],  ## Bounds for the p-norm p value (Minkowski) applied to mix the matrices
        'sig_NN_kwargs_mu': [0., 1.0],  ## Bounds for the sigmoid center for s_NN
        'sig_NN_kwargs_b': [0.1, 1.5],  ## Bounds for the sigmoid slope for s_NN
        'sig_SWT_kwargs_mu': [0., 1.0],  ## Bounds for the sigmoid center for s_SWT
        'sig_SWT_kwargs_b': [0.1, 1.5],  ## Bounds for the sigmoid slope for s_SWT
    },
    n_jobs_findParameters=-1,  ## Number of CPU cores to use (-1 is all cores)
    seed=SEED,  ## Random seed
)

#### Manual method

In [ ]:
# Uncomment below to manually specify mixing parameters
kwargs_makeConjunctiveDistanceMatrix_best = {
    'power_SF': 0.0,   ## s_sf**power_SF   (Higher values means clustering is more sensitive to spatial overlap of ROIs)
    'power_NN': 1.0,   ## s_NN**power_NN   (Higher values means clustering is more sensitive to visual similarity of ROIs)
    'power_SWT': 0.0,  ## s_SWT**power_SWT (Higher values means clustering is more sensitive to visual similarity of ROIs)
    'p_norm': -1,    ## norm([s_sf, s_NN, s_SWT], p=p_norm) (Higher values means clustering requires all similarity metrics to be high)
#     'sig_SF_kwargs': {'mu':0.5, 'b':1.0},  ## Sigmoid parameters for s_sf (mu is the center, b is the slope)
    'sig_SF_kwargs': None,
    # 'sig_NN_kwargs': {'mu':0.5, 'b':1},    ## Sigmoid parameters for s_NN (mu is the center, b is the slope)
    'sig_NN_kwargs': None,
#     'sig_NN_kwargs': None,
    'sig_SWT_kwargs': {'mu':0.5, 'b':3}, ## Sigmoid parameters for s_SWT (mu is the center, b is the slope)
#     'sig_SWT_kwargs': None,
}

#### View mixing results
The goal is to see a **bimodal curve** in the pairwise similarities and a **clear cross-over point** (specified by the vertical dotted line) between 'same' and 'diff' pairs of ROIs.

In [ ]:
clusterer.plot_distSame(kwargs_makeConjunctiveDistanceMatrix=kwargs_makeConjunctiveDistanceMatrix_best)

clusterer.plot_similarity_relationships(
    plots_to_show=[1,2,3], 
    max_samples=100000,  ## Make smaller if it is running too slow
    kwargs_scatter={'s':1, 'alpha':0.2},
    kwargs_makeConjunctiveDistanceMatrix=kwargs_makeConjunctiveDistanceMatrix_best
);

In [ ]:
clusterer.plot_distSame(kwargs_makeConjunctiveDistanceMatrix=kwargs_makeConjunctiveDistanceMatrix_best)

# clusterer.plot_similarity_relationships(
#     plots_to_show=[1,2,3], 
#     max_samples=100000,  ## Make smaller if it is running too slow
#     kwargs_scatter={'s':1, 'alpha':0.2},
#     kwargs_makeConjunctiveDistanceMatrix=kwargs_makeConjunctiveDistanceMatrix_best
# );

In [ ]:
clusterer.plot_distSame(kwargs_makeConjunctiveDistanceMatrix=kwargs_makeConjunctiveDistanceMatrix_best)

# clusterer.plot_similarity_relationships(
#     plots_to_show=[1,2,3], 
#     max_samples=100000,  ## Make smaller if it is running too slow
#     kwargs_scatter={'s':1, 'alpha':0.2},
#     kwargs_makeConjunctiveDistanceMatrix=kwargs_makeConjunctiveDistanceMatrix_best
# );

##### 2. Prune the distance matrix

We can remove all connections in the distance graph with probabilities of connection of less than 50%. We estimate this cutoff distance as the cross-over point between the 'same' and 'different' distributions.

**Important parameter**\
`stringency`: This value changes the threshold for pruning the distance matrix. A higher value will result in less pruning, and a lower value will result in more pruning. The value will be multiplied by the inferred threshold to get the new one.


In [ ]:
clusterer.make_pruned_similarity_graphs(
    d_cutoff=None,  ## Optionally manually specify a distance cutoff
    kwargs_makeConjunctiveDistanceMatrix=kwargs_makeConjunctiveDistanceMatrix_best,
    stringency=1.0,  ## Modifies the threshold for pruning the distance matrix. Higher values result in LESS pruning. New d_cutoff = stringency * truncated d_cutoff.
    convert_to_probability=False,    
)

##### 3. Cluster

There are two methods for clustering: The standard method `.fit` which is based on HDBSCAN, and `.fit_sequentialHungarian` which is an algorithm that is also used by CaImAn based on the Hungarian algorithm. The standard method takes 1-20 minutes and works better when there are many sessions, the Hungarian method takes seconds and works better when there are fewer sessions (<8).

**Important parameters**:
- For standard **`.fit`** method:
1. `min_cluster_size`: If you only want ROIs clusters with at least a certain number of samples, specify here.
2. `n_iter_violationCorrection`: This parameter controls how fast this step takes. Turning it down has mild effects on quality. We use around ***6***.

- For **`.fit_sequentialHungarian`** method:
1. `thresh_cost`: Determines the threshold of how distant two ROIs can be and still be matched. Smaller value is more stringent.


In [ ]:
# if data.n_sessions >= 6:
if True:
    labels = clusterer.fit(
        d_conj=clusterer.dConj_pruned,  ## Input distance matrix
        # session_bool=data.session_bool,  ## Boolean array of which ROIs belong to which sessions
        session_bool=sb,
        min_cluster_size=2,  ## Minimum number of ROIs that can be considered a 'cluster'
        n_iter_violationCorrection=6,  ## Number of times to redo clustering sweep after removing violations
        split_intraSession_clusters=True,  ## Whether or not to split clusters with ROIs from the same session
        cluster_selection_method='leaf',  ## (advanced) Method of cluster selection for HDBSCAN (see hdbscan documentation)
        d_clusterMerge=None,  ## Distance below which all ROIs are merged into a cluster
        alpha=0.999,  ## (advanced) Scalar applied to distance matrix in HDBSCAN (see hdbscan documentation)
        discard_failed_pruning=True,  ## (advanced) Whether or not to set all ROIs that could be separated from clusters with ROIs from the same sessions to label=-1
        n_steps_clusterSplit=100,  ## (advanced) How finely to step through distances to remove violations
    )

else:
    labels = clusterer.fit_sequentialHungarian(
        d_conj=clusterer.dConj_pruned,  ## Input distance matrix
        # session_bool=data.session_bool,  ## Boolean array of which ROIs belong to which sessions
        session_bool=sb,
        thresh_cost=0.8,  ## Threshold. Higher values result in more permissive clustering. Specifically, the pairwise metric distance between ROIs above which two ROIs cannot be clustered together.
    )

##### 4. Quality metrics

Compute various quality scores for each cluster and each ROI.

In [ ]:
## SKIP THIS STEP FOR VERY LARGE DATASETS
quality_metrics = clusterer.compute_quality_metrics();

## Collect results

1. Make different versions of the labels for convenience.
2. Put all the useful results and info into a dictionary to save later. ADJUST THIS ANY WAY YOU WANT.
3. Put all the class objects from the run into a dictionary to save later

In [ ]:
sb.shape

In [ ]:
# labels_squeezed, labels_bySession, labels_bool, labels_bool_bySession, labels_dict = roicat.tracking.clustering.make_label_variants(labels=labels, n_roi_bySession=data.n_roi)
labels_squeezed, labels_bySession, labels_bool, labels_bool_bySession, labels_dict = roicat.tracking.clustering.make_label_variants(labels=labels, n_roi_bySession=[s.sum(0) for s in sb.T])

results_clusters = {
    'labels': labels_squeezed,
    'labels_bySession': labels_bySession,
    'labels_dict': labels_dict,
    'quality_metrics': clusterer.quality_metrics if hasattr(clusterer, 'quality_metrics') else None,
}

results_all = {
    "clusters":{
        "labels": roicat.util.JSON_List(labels_squeezed),
        "labels_bySession": roicat.util.JSON_List(labels_bySession),
        "labels_bool": labels_bool,
        "labels_bool_bySession": labels_bool_bySession,
        "labels_dict": roicat.util.JSON_Dict(labels_dict),
        "quality_metrics": roicat.util.JSON_Dict(clusterer.quality_metrics) if hasattr(clusterer, 'quality_metrics') else None,
    },
    "ROIs": {
        # "ROIs_aligned": aligner.ROIs_aligned,
        "ROIs_aligned": r,
        # "ROIs_raw": data.spatialFootprints,
        # "frame_height": data.FOV_height,
        "frame_height": 512,
        # "frame_width": data.FOV_width,
        "frame_width": 1024,
        # "idx_roi_session": np.where(data.session_bool)[1],
        "idx_roi_session": np.where(sb)[1],
        # "n_sessions": data.n_sessions,
        "n_sessions": 35,
    },
    # "input_data": {
    #     "paths_stat": data.paths_stat,
    #     "paths_ops": data.paths_ops,
    # },
}

# run_data = {
#     'data': data.__dict__,
#     'aligner': aligner.__dict__,
#     'blurrer': blurrer.__dict__,
#     'roinet': roinet.__dict__,
#     'swt': swt.__dict__,
#     'sim': sim.__dict__,
#     'clusterer': clusterer.__dict__,
# }

# params_used = {name: mod['params'] for name, mod in run_data.items()}

# Visualize results

In [ ]:
print(f'Number of clusters: {len(np.unique(results_clusters["labels"]))}')
print(f'Number of discarded ROIs: {(np.array(results_clusters["labels"])==-1).sum()}')

Look at some of the distributions of the quality metrics.
- Silhouette score is a particularly useful one for this type of clustering. Learn more here: https://en.wikipedia.org/wiki/Silhouette_(clustering)
- We also define a handy 'confidence' variable which is a nice heuristic you can use for thresholding for inclusion criteria
- Note that the `sample_silhouette` score is a per-sample (per-ROI) score. So it can actually be used to remove / subselect ROIs from clusters.

A good rule of thumb is to use an inclusion criteria of:
- `sample_silhouette` > 0.1
- `cluster_silhouette` > 0.2

In [ ]:
roicat.tracking.clustering.plot_quality_metrics(quality_metrics=quality_metrics, labels=labels_squeezed, n_sessions=35);

Look at a color visualization of the results. ROIs of the same color are considered a part of the same cluster. The colors are assigned randomly.

In [ ]:
FOV_clusters = roicat.visualization.compute_colored_FOV(
    spatialFootprints=[r.power(1.0) for r in results_all['ROIs']['ROIs_aligned']],  ## Spatial footprint sparse arrays
    FOV_height=results_all['ROIs']['frame_height'],
    FOV_width=results_all['ROIs']['frame_width'],
    labels=results_all["clusters"]["labels_bySession"],  ## cluster labels
#     labels=(np.array(results["clusters"]["labels"])!=-1).astype(np.int64),  ## cluster labels
    # alphas_labels=confidence*1.5,  ## Set brightness of each cluster based on some 1-D array
    # alphas_labels=(np.array(clusterer.quality_metrics['cluster_silhouette']) > -0.2) * (np.array(clusterer.quality_metrics['cluster_intra_means']) > 0.3),
    # alphas_sf=np.array(clusterer.quality_metrics['sample_silhouette']) > 0,  ## Set brightness of each ROI based on some 1-D array
)

roicat.visualization.display_toggle_image_stack(
    FOV_clusters, 
    image_size=1.5,
#     clim=[0,1.0],
)

Visualize the images of ROIs from the same cluster

In [ ]:
roicat.visualization.display_cropped_cluster_ims(
    spatialFootprints=results_all['ROIs']['ROIs_aligned'],
    labels=np.array(results_all["clusters"]["labels"]),
    FOV_height=results_all['ROIs']['frame_height'],
    FOV_width=results_all['ROIs']['frame_width'],
    n_labels_to_display=10,    
)

# Save results

In [ ]:
## Define the directory to save the results to
dir_save = '/media/rich/bigSSD/data_tmp/test_data/'
name_save = 'mouse_1'

paths_save = {
    'results_clusters': str(Path(dir_save) / f'{name_save}.tracking.results_clusters.json'),
    'params_used':      str(Path(dir_save) / f'{name_save}.tracking.params_used.json'),
    'results_all':      str(Path(dir_save) / f'{name_save}.tracking.results_all.richfile'),
    'run_data':         str(Path(dir_save) / f'{name_save}.tracking.run_data.richfile'),
}


In [ ]:
Path(dir_save).mkdir(parents=True, exist_ok=True)

roicat.helpers.json_save(obj=results_clusters, filepath=paths_save['results_clusters'])
roicat.helpers.json_save(obj=params_used, filepath=paths_save['params_used'])
roicat.util.RichFile_ROICaT(path=paths_save['results_all']).save(obj=results_all, overwrite=True)
roicat.util.RichFile_ROICaT(path=paths_save['run_data']).save(obj=run_data, overwrite=True)

Optionally save the FOV_clusters images as a GIF

In [ ]:
roicat.helpers.save_gif(
    array=roicat.helpers.add_text_to_images(
        images=[(f * 255).astype(np.uint8) for f in FOV_clusters], 
        text=[[f"{ii}",] for ii in range(len(FOV_clusters))], 
        font_size=3,
        line_width=10,
        position=(30, 90),
    ), 
    path=str(Path(dir_save).resolve() / 'visualization' / 'FOV_clusters_highQuality.gif'),
    frameRate=10.0,
    loop=0,
)

# Extras

##### Demo: Importing richfiles

Simple output files are saved as .json files, which are easily handled. However, complex output files are saved using a custom format called `richfile`, which are basically just structured directories containing data files. This is a demonstration of how to import and handle richfile directories.

In [ ]:
## Make a richfile object of the results_all dictionary
r = roicat.util.RichFile_ROICaT(path=paths_save['results_all'])

## You can load it
temp_results_all = r.load()

## You can load part of it by indexing into the richfile object
print(f"n_sessions: {r['ROIs']['n_sessions'].load()}")
print('')

## You can view the tree structure
r.view_tree()

#### Look at `params_used`
These were the parameters that were used in the run, they can be used to reproduce the run using the CLI by creating a params.yaml file.


In [ ]:
params_used

##### Legacy saving methods

In [ ]:
dir_save = Path('/media/rich/bigSSD/data_tmp/test_data/').resolve()
name_save = Path(dir_allOuterFolders).resolve().name

path_save = dir_save / (name_save + '.ROICaT.tracking.results' + '.pkl')
print(f'path_save: {path_save}')

In [ ]:
run_data_sd = {
    'data': data.serializable_dict,
    'aligner': aligner.serializable_dict,
    'blurrer': blurrer.serializable_dict,
    'roinet': roinet.serializable_dict,
    'swt': swt.serializable_dict,
    'sim': sim.serializable_dict,
    'clusterer': clusterer.serializable_dict,
}

In [ ]:
roicat.helpers.pickle_save(
    obj=results_all,
    filepath=path_save,
    mkdir=True,
)

roicat.helpers.pickle_save(
    obj=run_data_sd,
    filepath=str(dir_save / (name_save + '.ROICaT.tracking.rundata' + '.pkl')),
    mkdir=True,
)


Optionally save results as a matlab file.

In [ ]:
roicat.helpers.matlab_save(
    obj=results_all,
    filepath=str(dir_save / (name_save + '.ROICaT.tracking.results' + '.mat')),
)

## Save more figures

In [ ]:
### Save gifs of the FOVs at different stages of alignment
roicat.helpers.save_gif(
    array=roicat.helpers.add_text_to_images(
        images=[((f / np.max(f)) * 255).astype(np.uint8) for f in FOV_images], 
        text=[[f"{ii}",] for ii in range(len(FOV_clusters))], 
        font_size=3,
        line_width=10,
        position=(30, 90),
    ), 
    path=str(Path(dir_save).resolve() / 'visualization' / 'FOV_images.gif'),
    frameRate=10,
    loop=0,
)

roicat.helpers.save_gif(
    array=roicat.helpers.add_text_to_images(
        images=[((f / np.max(f)) * 255).astype(np.uint8) for f in aligner.ims_registered_geo], 
        text=[[f"{ii}",] for ii in range(len(FOV_clusters))], 
        font_size=3,
        line_width=10,
        position=(30, 90),
    ), 
    path=str(Path(dir_save).resolve() / 'visualization' / 'FOV_images_aligned_geometric.gif'),
    frameRate=10,
    loop=0,
)

roicat.helpers.save_gif(
    array=roicat.helpers.add_text_to_images(
        images=[((f / np.max(f)) * 255).astype(np.uint8) for f in aligner.ims_registered_nonrigid], 
        text=[[f"{ii}",] for ii in range(len(FOV_clusters))], 
        font_size=3,
        line_width=10,
        position=(30, 90),
    ), 
    path=str(Path(dir_save).resolve() / 'visualization' / 'FOV_images_aligned_nonrigid.gif'),
    frameRate=10,
    loop=0,
)

roicat.helpers.save_gif(
    array=roicat.helpers.add_text_to_images(
        images=[((f / np.max(f)) * 255).astype(np.uint8) for f in aligner.get_ROIsAligned_maxIntensityProjection(normalize=True)], 
        text=[[f"{ii}",] for ii in range(len(FOV_clusters))], 
        font_size=3,
        line_width=10,
        position=(30, 90),
    ), 
    path=str(Path(dir_save).resolve() / 'visualization' / 'FOV_ROIs_aligned.gif'),
    frameRate=10,
    loop=0,
)